# Notebook 03 — Feature Engineering
## NBA Contract Value Index (CVI)

This notebook takes our cleaned master dataset (stats + salary) and transforms 
the raw numbers into 11 CVI feature scores (0–100) across 9 real features
and 2 v2 placeholders. Outputs two final CVI scores — one from the team
perspective and one from the player perspective.

Each score feeds into the logistic regression model in notebook 05.
A higher score = better value for that feature.
The weighted combination of all scores = the final CVI probability.

---

### CVI Feature Scorecard

| # | Feature | Team Weight | Player Weight | Source | Method | Status |
|---|---------|-------------|---------------|--------|--------|--------|
| 1 | Win impact | 18.1% | 22.0% | BPM, VORP, WS | Peer-normalized within 4-tier salary structure | ✓ Real |
| 2 | Availability | 15.2% | 18.0% | G | 3-yr weighted rolling avg (50/30/20) | ✓ Real |
| 3 | Market comparison | 13.3% | 16.0% | SALARY_M, BPM, AGE | OLS wage regression residual | ✓ Real |
| 4 | Age / trajectory | 11.4% | 13.0% | AGE, POSITION_GROUP, YEARS_REMAINING | Gaussian decay + years remaining penalty | ✓ Real |
| 5 | Cap efficiency | 10.5% | 12.0% | SALARY_M, BPM | BPM tier percentile minus salary tier percentile | ✓ Real |
| 6 | Role / minutes fit | 9.5% | 10.0% | MIN, SALARY_TIER | Actual vs expected minutes by tier | ✓ Real |
| 7 | Apron & tax impact | 8.6% | 5.0% | YEARS_REMAINING, SALARY_M | Salary + years remaining penalty structure | ✓ Real |
| 8 | Team payroll context | 8.6% | 0.0% | TEAM_PAYROLL_M, APRON_STATUS | Real Spotrac apron data + payroll share % | ✓ Real |
| 9 | Accolades | 4.8% | 4.0% | Career awards | Weighted decay (half-life 5yr) — 35 players | ✓ Real |
| 10 | Marketability | v2 | v2 | Google Trends, social followers | Placeholder 50.0 | ⏳ V2 |
| 11 | Jersey sales | v2 | v2 | NBA Store rankings | Placeholder 50.0 | ⏳ V2 |

---

### Dual CVI Score System

| Score | Perspective | Question it answers |
|-------|-------------|---------------------|
| `CVI_SCORE` | Team | Is this contract good for the team's roster construction? |
| `CVI_PLAYER_SCORE` | Player | Is this player worth what they're being paid individually? |
| `CVI_GAP` | Difference | How much is team context helping or hurting the contract? |

**Gap interpretation:**
- Negative gap → good player trapped in bad team cap situation (Mitchell, Mobley)
- Near zero gap → team and player value aligned (SGA, Jokić)
- Positive gap → team situation making contract look better than it is (LeBron, Middleton)

Key finding: CLE players (Mitchell -1.6, Mobley -4.3, Allen -4.2) all show
negative gaps driven by Cleveland's second apron status — not the players' fault.

---

### Salary Tier Structure (4 tiers)

| Tier | Criteria | Description |
|------|----------|-------------|
| Franchise | Max contract + top 12 BPM + age ≤ 32 + 40+ games | True cornerstones |
| Max | 30M+ but not franchise tier | Solid max players |
| Star | 15M–30M | Quality starters |
| Role | Under 15M | Role players and backups |

---

### Flags Added

| Flag | Definition | Purpose |
|------|-----------|---------|
| `UNDERVALUED_FLAG` | Star tier + top 20 BPM | Brunson-type underpaid franchise performers |
| `OVERPAID_MAX_FLAG` | Max tier but not franchise tier | Paid like a cornerstone, not performing like one |
| `PLAYMAKER_FLAG` | AST% ≥ 25 + BPM ≥ 0 + star/max tier | Guards whose BPM undersells playmaking value |
| `ROOKIE_SCALE_FLAG` | Age ≤ 24 + salary < $20M + role tier | Structurally underpaid young players |

---

### Key Decisions Made in This Notebook

- Rebuilt master from scratch (dedup → salary merge → contract merge) for full independence
- 4-tier salary structure — separates true franchise players from overpaid max contracts
- Franchise tier requires performance (top 12 BPM) + availability (40+ games) + age (≤32)
- OLS market comparison trained on training data only — prevents data leakage
- Cap efficiency v2 redesign — BPM tier percentile minus salary tier percentile
  (v1 ratio approach broke at salary extremes, Jokić scored 0.0)
- Apron status for 2025-26 uses real Spotrac data, not ESPN estimates
- Team payroll calculated from ESPN salary sums — slightly understated but consistent
- Accolades manually curated for 35 key players — v2 automates via BBRef scraping
- PAYROLL_CONTEXT_SCORE excluded from player score — team cap issues aren't player's fault
- APRON_SCORE reduced from 8.6% to 5.0% in player score for same reason

---

### Data Flow

data/raw/bbref_master.csv            ← notebook 01 output (BBRef stats)

data/raw/espn_salaries.csv           ← notebook 02 output (ESPN salary history)

data/raw/bbref_contracts_wide.csv    ← notebook 02 output (current contracts)

data/raw/spotrac_apron_2526.csv      ← manually entered from Spotrac


data/processed/master_with_salary.csv    ← rebuilt + cleaned checkpoint


data/processed/master_with_features.csv  ← final output (this notebook)

---

### Output Columns Added This Notebook

| Column | Description |
|--------|-------------|
| `SALARY_TIER` | franchise / max / star / role / unknown |
| `WIN_IMPACT_SCORE` | 0–100, peer-normalized within tier |
| `AVAILABILITY_SCORE` | 0–100, 3-yr weighted rolling GP% |
| `MARKET_SCORE` | 0–100, OLS residual (underpaid = high) |
| `AGE_SCORE` | 0–100, Gaussian decay by position |
| `CAP_EFFICIENCY_SCORE` | 0–100, BPM tier percentile minus salary tier percentile |
| `ROLE_FIT_SCORE` | 0–100, actual vs expected minutes |
| `APRON_SCORE` | 0–100, salary + years remaining penalty |
| `PAYROLL_CONTEXT_SCORE` | 0–100, team apron status + payroll share |
| `ACCOLADES_SCORE` | 0–100, decayed career award points |
| `MARKETABILITY_SCORE` | 50.0 placeholder (v2) |
| `JERSEY_SALES_SCORE` | 50.0 placeholder (v2) |
| `CONTRACT_TYPE_SCORE` | 50.0 placeholder (v2) |
| `INJURY_RISK_SCORE` | 50.0 placeholder (v2) |
| `UNDERVALUED_FLAG` | 1 if star tier + top 20 BPM |
| `OVERPAID_MAX_FLAG` | 1 if max tier but not franchise |
| `PLAYMAKER_FLAG` | 1 if high AST% + positive BPM |
| `ROOKIE_SCALE_FLAG` | 1 if age ≤ 24 + salary < $20M + role tier |
| `TEAM_WINPCT_DELTA` | NaN placeholder — calculated in notebook 05 |
| `CVI_SCORE` | 0–100, team perspective composite |
| `CVI_PLAYER_SCORE` | 0–100, player perspective composite |
| `CVI_GAP` | team score minus player score |
| `CVI_VERDICT` | good / borderline / bad (tiered thresholds) |
| `CVI_PLAYER_VERDICT` | good / borderline / bad (player perspective) |

---

### CVI Score Results Summary — 2025-26

| Metric | Value |
|--------|-------|
| Mean CVI | 59.5 |
| Median CVI | 59.8 |
| Max CVI | 82.9 (Payton Pritchard) |
| Min CVI | 31.9 (Khris Middleton) |
| Good contracts | 184 |
| Borderline | 233 |
| Bad contracts | 33 |

Top franchise verdicts: SGA (79.9 good), Jokić (71.0 good),
Maxey (58.1 borderline), Luka (54.1 borderline),
Cade (48.3 borderline), Mitchell (42.3 borderline→bad)

---

### Notes for V2
- Automate accolades via BBRef awards page scraping
- Add Google Trends + SocialBlade for marketability
- Add NBA Store jersey rankings for jersey sales score
- Add contract type (player option, team option) from Spotrac
- Add injury risk model using prosportstransactions.com data
- Scrape Spotrac historical apron data for 2021-22 through 2024-25
- Salary prediction model — predict what players SHOULD make (regression)
- Refine market comparison to account for age-at-signing
  (Luka signed young — OLS doesn't fully capture this)
- Consider separate CVI model for each salary tier

### Imports

In [49]:
import pandas as pd
import numpy as np
import warnings
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
import unicodedata
import time
import requests

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

print("✓ Imports ready")

✓ Imports ready


### Load all CSV's

In [23]:
# Load everything from saved CSVs
# Notebook 03 is fully independent — no need for other notebooks
# to be open or variables to be in memory

bbref_master    = pd.read_csv("../data/raw/bbref_master.csv")
espn_clean      = pd.read_csv("../data/raw/espn_salaries.csv")
contracts_wide  = pd.read_csv("../data/raw/bbref_contracts_wide.csv")
contracts_long  = pd.read_csv("../data/raw/bbref_contracts_long.csv")

print(f"✓ BBRef master:      {len(bbref_master)} rows")
print(f"✓ ESPN salaries:     {len(espn_clean)} rows")
print(f"✓ BBRef contracts:   {len(contracts_wide)} players")
print(f"✓ Contracts long:    {len(contracts_long)} rows")

✓ BBRef master:      2663 rows
✓ ESPN salaries:     2466 rows
✓ BBRef contracts:   527 players
✓ Contracts long:    1242 rows


### Data loaded
All 4 source files loaded from CSV — notebook 03 is fully independent.
No other notebook needs to be open or have been run in the same session.
This is the correct pattern for every notebook going forward.

Next: define the functions that will be used throughout this notebook
before we touch any data transformations.

### Name fix functions

In [24]:
def fix_player_names(name):
    """
    Convert accented unicode characters to ASCII equivalents.
    Handles encoding issues from BBRef HTML scraping.
    """
    if pd.isna(name):
        return name
    return (unicodedata.normalize("NFKD", str(name))
            .encode("ascii", "ignore")
            .decode("utf-8")
            .strip())

# Every name mismatch we found across notebooks 01 and 02
name_fixes_exact = {
    # Broken unicode
    "Nikola JokiA"            : "Nikola Jokic",
    "Luka DonAiA"             : "Luka Doncic",
    "Bogdan BogdanoviA"       : "Bogdan Bogdanovic",
    "Bojan BogdanoviA"        : "Bojan Bogdanovic",
    "Boban MarjanoviA"        : "Boban Marjanovic",
    "Goran DragiA"            : "Goran Dragic",
    "Jusuf NurkiA"            : "Jusuf Nurkic",
    "Nikola JoviA"            : "Nikola Jovic",
    "Nikola VuAeviA"          : "Nikola Vucevic",
    "Vasilije MiciA"          : "Vasilije Micic",
    "Karlo MatkoviA"          : "Karlo Matkovic",
    "Kristaps PorziAAis"      : "Kristaps Porzingis",
    "Dario A ariA"            : "Dario Saric",
    "Luka A amaniA"           : "Luka Samanic",
    "DAvis BertAns"           : "Davis Bertans",
    "Egor DNmin"              : "Egor Demin",
    "TomAA SatoranskA12"      : "Tomas Satoransky",
    "VAt KrejAA"              : "Vit Krejci",
    "Alperen AengA14n"        : "Alperen Sengun",
    "Dennis SchrAder"         : "Dennis Schroder",
    "Moussa DiabatA"          : "Moussa Diabate",
    "Jonas ValanAiAnas"       : "Jonas Valanciunas",
    "Tidjane SalaA14n"        : "Tidjane Salaun",
    "Kasparas JakuAionis"     : "Kasparas Jakucionis",
    "Hugo GonzAlez"           : "Hugo Gonzalez",
    "PacA me Dadiet"          : "Pacome Dadiet",
    "Yanic Konan NiederhAuser": "Yanic Konan Niederhauser",
    "Nolan TraorA"            : "Nolan Traore",
    # Suffix mismatches
    "Michael Porter Jr."      : "Michael Porter",
    "Gary Payton II"          : "Gary Payton",
    "Jaren Jackson Jr."       : "Jaren Jackson",
    "Wendell Carter Jr."      : "Wendell Carter",
    "Derrick Jones Jr."       : "Derrick Jones",
    "Larry Nance Jr."         : "Larry Nance",
    "Kevin Porter Jr."        : "Kevin Porter",
    "Ron Harper Jr."          : "Ron Harper",
    "Nick Smith Jr."          : "Nick Smith",
    "Kevin McCullar Jr."      : "Kevin McCullar",
    "GG Jackson II"           : "GG Jackson",
    "Robert Williams"         : "Robert Williams III",
    "Nic Claxton"             : "Nicolas Claxton",
    "Mo Wagner"               : "Moritz Wagner",
}

additional_name_fixes = {
    "Michael Porter Jr."      : "Michael Porter",
    "Gary Payton II"          : "Gary Payton",
    "Jaren Jackson Jr."       : "Jaren Jackson",
    "Wendell Carter Jr."      : "Wendell Carter",
    "Derrick Jones Jr."       : "Derrick Jones",
    "Larry Nance Jr."         : "Larry Nance",
    "Kevin Porter Jr."        : "Kevin Porter",
    "Ron Harper Jr."          : "Ron Harper",
    "GG Jackson II"           : "GG Jackson",
    "Robert Williams"         : "Robert Williams III",
    "Nic Claxton"             : "Nicolas Claxton",
}

final_name_fixes = {
    "Tristan Da Silva"        : "Tristan da Silva",
    "Pacôme Dadiet"           : "Pacome Dadiet",
}

print("✓ Name fix functions defined")

✓ Name fix functions defined


### Name fix functions defined
Two layers of name fixing:
1. `fix_player_names()` — unicode normalization (handles most accented chars)
2. `name_fixes_exact` dict — manual overrides for names that broke differently

- We carry ALL fixes from notebooks 01 and 02 here so this notebook
is the single source of truth for name standardization.
The same fixes are applied to both BBRef and ESPN data before any joins.

### Helper Functions

In [25]:
def dedup_traded_players(df):
    """
    BBRef creates 2TM/3TM rows for players traded mid-season.
    These rows contain full season totals — keep them and drop
    the individual team split rows.
    
    Teaching note:
    Without this step a traded player like Luka Doncic shows up
    3 times in 2024-25 (DAL row, LAL row, 2TM row). That would
    triple-count his stats and distort every feature calculation.
    """
    traded = df[df["TEAM"].isin(["2TM","3TM"])][
        ["PLAYER_NAME","SEASON"]
    ].drop_duplicates()
    traded["IS_TRADED"] = True
    
    df = df.merge(traded, on=["PLAYER_NAME","SEASON"], how="left")
    
    df_clean = df[
        (df["IS_TRADED"] != True) |
        (df["TEAM"].isin(["2TM","3TM"]))
    ].copy()
    
    df_clean = df_clean.drop(columns=["IS_TRADED"])
    df_clean = df_clean.reset_index(drop=True)
    
    print(f"  Traded players:    {len(traded)}")
    print(f"  Rows before dedup: {len(df)}")
    print(f"  Rows after dedup:  {len(df_clean)}")
    return df_clean


def merge_espn_salary(master_df, espn_df):
    """
    Left join ESPN salary onto master by PLAYER_NAME + SEASON.
    Left join keeps all players even without a salary match.
    Unmatched players get NaN salary — handled in feature engineering.
    """
    cols_to_drop = ["SALARY","SALARY_M","SALARY_THIS_YEAR_M",
                    "SALARY_TOTAL_M","YEARS_REMAINING","SALARY_AVG_PER_YR_M"]
    master_df = master_df.drop(
        columns=[c for c in cols_to_drop if c in master_df.columns]
    ).copy()
    
    merged = master_df.merge(
        espn_df[["PLAYER_NAME","SEASON","SALARY","SALARY_M"]],
        on  = ["PLAYER_NAME","SEASON"],
        how = "left"
    )
    
    matched = merged["SALARY_M"].notna().sum()
    total   = len(merged)
    print(f"  Salary matched: {matched}/{total} ({matched/total*100:.1f}%)")
    return merged


def assign_salary_tier(df):
    """
    4-tier salary classification.
    Franchise = max contract + top 12 BPM + age ≤ 32 + 40+ games.
    Max     = $30M+ but not franchise tier
    Star    = $15M-$30M
    Role    = under $15M
    """
    df = df.copy()
    
    def base_tier(salary):
        if pd.isna(salary):   return "unknown"
        elif salary >= 30:    return "max"
        elif salary >= 15:    return "star"
        else:                 return "role"
    
    df["SALARY_TIER"] = df["SALARY_M"].apply(base_tier)
    
    for season in df["SEASON"].unique():
        season_mask = df["SEASON"] == season
        max_mask    = df["SALARY_TIER"] == "max"
        games_mask  = df["G"] >= 40
        age_mask    = df["AGE"] <= 32
        
        top12_bpm = df[season_mask]["BPM"].nlargest(12).min()
        
        franchise_mask = (
            season_mask & max_mask &
            games_mask  & age_mask &
            (df["BPM"] >= top12_bpm)
        )
        df.loc[franchise_mask, "SALARY_TIER"] = "franchise"
    
    return df

print("✓ Helper functions defined")

✓ Helper functions defined


### Helper functions defined
Three core functions that build the master table:

- `dedup_traded_players()` — removes individual team split rows,
  keeps only 2TM/3TM full-season totals for traded players
- `merge_espn_salary()` — left joins salary onto master by player + season
- `assign_salary_tier()` — 4-tier classification (franchise/max/star/role)

Franchise tier criteria: max contract + top 12 BPM + age ≤ 32 + 40+ games.
This separates true cornerstones (Jokić, SGA, Luka) from players who
got paid like one but don't perform like one.

These functions are defined once and reused — if logic needs to change
we update the function, not every place it's called.

### Rebuild Master

In [26]:
# Rebuild master cleanly from saved CSVs
# Every transformation step in one place — easy to follow and debug

print("Step 1 — dedup traded players...")
master = dedup_traded_players(bbref_master.copy())

print("\nStep 2 — fix player names...")
master["PLAYER_NAME"] = master["PLAYER_NAME"].apply(fix_player_names)
master["PLAYER_NAME"] = master["PLAYER_NAME"].replace(name_fixes_exact)
master["PLAYER_NAME"] = master["PLAYER_NAME"].replace(additional_name_fixes)
master["PLAYER_NAME"] = master["PLAYER_NAME"].replace(final_name_fixes)
espn_clean["PLAYER_NAME"] = espn_clean["PLAYER_NAME"].replace(additional_name_fixes)
espn_clean["PLAYER_NAME"] = espn_clean["PLAYER_NAME"].replace(final_name_fixes)
print("  ✓ Names fixed")

print("\nStep 3 — merge ESPN salary...")
master = merge_espn_salary(master, espn_clean)

print("\nStep 4 — merge BBRef contract metrics...")
salary_cols = [c for c in contracts_wide.columns if c.startswith("SALARY_20")]

contract_summary = contracts_wide.copy()
contract_summary["SALARY_TOTAL_M"] = (
    contracts_wide[salary_cols].sum(axis=1, skipna=True) / 1_000_000
).round(2)
contract_summary["YEARS_REMAINING"] = (
    contracts_wide[salary_cols].apply(lambda r: (r > 0).sum(), axis=1)
)
contract_summary["SALARY_AVG_PER_YR_M"] = np.where(
    contract_summary["YEARS_REMAINING"] > 0,
    contract_summary["SALARY_TOTAL_M"] / contract_summary["YEARS_REMAINING"],
    np.nan
)

# Fix names in contract summary
contract_summary["PLAYER_NAME"] = (contract_summary["PLAYER_NAME"]
                                   .replace(name_fixes_exact)
                                   .replace(additional_name_fixes))

master = master.merge(
    contract_summary[["PLAYER_NAME","TEAM",
                       "SALARY_TOTAL_M","YEARS_REMAINING",
                       "SALARY_AVG_PER_YR_M"]],
    on  = ["PLAYER_NAME","TEAM"],
    how = "left"
)
print(f"  ✓ Contract metrics merged")

print("\nStep 5 — tag training vs current...")
master["DATA_TYPE"] = master["SEASON"].apply(
    lambda s: "current" if s == "2025-26" else "training"
)

print("\nStep 6 — assign salary tiers...")
master = assign_salary_tier(master)

print(f"\n✓ Master rebuilt: {len(master)} rows")
print(f"\nRows per season:")
print(master.groupby(["SEASON","DATA_TYPE"])["PLAYER_NAME"]
      .count().to_string())
print(f"\nTier counts (all seasons):")
print(master["SALARY_TIER"].value_counts().to_string())

Step 1 — dedup traded players...
  Traded players:    325
  Rows before dedup: 2663
  Rows after dedup:  2230

Step 2 — fix player names...
  ✓ Names fixed

Step 3 — merge ESPN salary...
  Salary matched: 2071/2230 (92.9%)

Step 4 — merge BBRef contract metrics...
  ✓ Contract metrics merged

Step 5 — tag training vs current...

Step 6 — assign salary tiers...

✓ Master rebuilt: 2233 rows

Rows per season:
SEASON   DATA_TYPE
2021-22  training     444
2022-23  training     436
2023-24  training     445
2024-25  training     456
2025-26  current      452

Tier counts (all seasons):
SALARY_TIER
role         1590
star          274
max           176
unknown       160
franchise      33


### Master table rebuilt
Clean 6-step pipeline:
1. Dedup traded players (2TM/3TM rows)
2. Fix player name encoding issues
3. Merge ESPN annual salary (target: 93%+ match rate)
4. Merge BBRef contract summary metrics (total value, years remaining)
5. Tag training vs current season
6. Assign 4-tier salary structure

Check the tier counts before moving on — franchise should be 5–10 per season,
max 30–50, star 50–80, role 300+. If numbers look off something went wrong
in the merge and we fix it before touching features.

### Save Checkpoint

In [27]:
# Save master_with_salary — clean checkpoint before feature engineering
# If anything breaks in features we reload from here, not from scratch
os.makedirs("../data/processed", exist_ok=True)
master.to_csv("../data/processed/master_with_salary.csv", index=False)

print(f"✓ Checkpoint saved — {len(master)} rows")
print(f"  → data/processed/master_with_salary.csv")
print(f"\nColumns ({master.shape[1]} total):")
print(master.columns.tolist())

✓ Checkpoint saved — 2233 rows
  → data/processed/master_with_salary.csv

Columns (61 total):
['PLAYER_NAME', 'AGE', 'TEAM', 'POSITION', 'G', 'GAMES_STARTED', 'MIN', 'FG', 'FGA', 'FG_PCT', '3P', '3PA', '3P_PCT', '2P', '2PA', '2P_PCT', 'EFG_PCT', 'FT', 'FTA', 'FT_PCT', 'ORB', 'DRB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'AWARDS', 'SEASON', 'PER', 'TS_PCT', '3PAR', 'FTR', 'ORB_PCT', 'DRB_PCT', 'TRB_PCT', 'AST_PCT', 'STL_PCT', 'BLK_PCT', 'TOV_PCT', 'USG_PCT', 'OWS', 'DWS', 'WS', 'WS_48', 'OBPM', 'DBPM', 'BPM', 'VORP', 'LAST_TEAM', 'DATA_TYPE', 'POSITION_GROUP', 'PLAYER_NAME_RAW', 'SALARY', 'SALARY_M', 'SALARY_TOTAL_M', 'YEARS_REMAINING', 'SALARY_AVG_PER_YR_M', 'SALARY_TIER']


### Sanity Check

In [28]:
# Quick check — why 2233 instead of 2230?
# Contract metrics merge likely created 3 duplicate rows
# Check for any player-season duplicates

dupes = master[master.duplicated(subset=["PLAYER_NAME","SEASON"], keep=False)]
print(f"Duplicate player-seasons: {len(dupes)}")
if len(dupes) > 0:
    print(dupes[["PLAYER_NAME","TEAM","SEASON","SALARY_M"]]
          .sort_values(["PLAYER_NAME","SEASON"])
          .head(20)
          .to_string(index=False))

Duplicate player-seasons: 6
    PLAYER_NAME TEAM  SEASON  SALARY_M
 Charles Bassey  PHI 2021-22      0.93
 Charles Bassey  PHI 2021-22      0.93
  Killian Hayes  SAC 2025-26       NaN
  Killian Hayes  SAC 2025-26       NaN
Pat Connaughton  CHO 2025-26      1.32
Pat Connaughton  CHO 2025-26      1.32


In [29]:
# Fix — drop duplicates keeping first row
before = len(master)
master = master.drop_duplicates(
    subset=["PLAYER_NAME","SEASON"],
    keep="first"
).reset_index(drop=True)

print(f"Before: {before} rows")
print(f"After:  {len(master)} rows")
print(f"Removed: {before - len(master)} duplicates")

# Resave clean checkpoint
master.to_csv("../data/processed/master_with_salary.csv", index=False)
print(f"\n✓ Clean checkpoint resaved — {len(master)} rows")

Before: 2233 rows
After:  2230 rows
Removed: 3 duplicates

✓ Clean checkpoint resaved — 2230 rows


### Master confirmed clean
- 2230 rows, 0 duplicates
- 92.9% salary coverage
- 5 seasons: 2021-22 through 2025-26
- 4 salary tiers assigned correctly


### Checkpoint saved → data/processed/master_with_salary.csv

This is our safety net. If anything breaks during feature engineering
we reload from this file instead of rebuilding from scratch.

Rule: always save a checkpoint before a major transformation step.
Features are calculated on top of this file — they never modify the
underlying stats or salary data, only add new columns.

---

## Feature Engineering begins here

We now calculate 10 CVI scores (0–100) for every player-season.
Each feature is independent — a bug in Feature 3 doesn't affect Feature 1.
Features are calculated in order of weight (highest → lowest impact).

Reminder of the scoring logic:
- 100 = best possible value for that feature
- 0   = worst possible value
- All scores normalized within season to avoid year-to-year salary inflation skew

In [33]:
def calculate_win_impact_score(df):
    """
    Feature 1 — Win Impact Score (Weight: 18%)
    
    Composite of BPM (40%), VORP (35%), WS (25%).
    Normalized within salary tier so max players are compared
    to other max players, not role players.
    
    Teaching note — why composite?
    BPM rewards high-usage scorers on bad teams.
    VORP penalizes players who miss games.
    WS rewards players on good teams.
    Combining all three balances their individual blind spots.
    """
    df = df.copy()
    
    # Build composite metric
    df["WIN_IMPACT_RAW"] = (
        df["BPM"]  * 0.40 +
        df["VORP"] * 0.35 +
        df["WS"]   * 0.25
    )
    
    # Normalize within each salary tier
    df["WIN_IMPACT_SCORE"] = np.nan
    
    for tier in ["franchise", "max", "star", "role"]:
        mask   = df["SALARY_TIER"] == tier
        values = df.loc[mask, "WIN_IMPACT_RAW"].values.reshape(-1, 1)
        if len(values) < 2:
            continue
        scaler = MinMaxScaler((0, 100))
        df.loc[mask, "WIN_IMPACT_SCORE"] = (
            scaler.fit_transform(values).flatten()
        )
    
    # Unknown tier — league-wide normalization
    unknown_mask = df["SALARY_TIER"] == "unknown"
    if unknown_mask.sum() > 0:
        values = df.loc[unknown_mask, "WIN_IMPACT_RAW"].values.reshape(-1, 1)
        scaler = MinMaxScaler((0, 100))
        df.loc[unknown_mask, "WIN_IMPACT_SCORE"] = (
            scaler.fit_transform(values).flatten()
        )
    
    df["WIN_IMPACT_SCORE"] = df["WIN_IMPACT_SCORE"].round(1)
    
    print(f"✓ Win impact scores calculated")
    print(f"\nScore distribution by tier (2025-26):")
    for tier in ["franchise", "max", "star", "role"]:
        tier_data = df[
            (df["SEASON"] == "2025-26") &
            (df["SALARY_TIER"] == tier)
        ]["WIN_IMPACT_SCORE"]
        if len(tier_data) > 0:
            print(f"  {tier:12s}: avg={tier_data.mean():.1f}  "
                  f"min={tier_data.min():.1f}  "
                  f"max={tier_data.max():.1f}")
    
    print(f"\nTop 10 win impact — franchise tier 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_TIER"] == "franchise")
    ][["PLAYER_NAME","TEAM","BPM","VORP","WS","WIN_IMPACT_SCORE"]]
    .sort_values("WIN_IMPACT_SCORE", ascending=False)
    .head(10)
    .to_string(index=False))
    
    return df

master = calculate_win_impact_score(master)

✓ Win impact scores calculated

Score distribution by tier (2025-26):
  franchise   : avg=40.1  min=7.6  max=90.3
  max         : avg=47.0  min=0.0  max=88.0
  star        : avg=39.8  min=6.1  max=68.3
  role        : avg=44.9  min=16.5  max=100.0

Top 10 win impact — franchise tier 2025-26:
            PLAYER_NAME TEAM  BPM  VORP   WS  WIN_IMPACT_SCORE
           Nikola Jokic  DEN 14.1   9.0 14.6              90.3
Shai Gilgeous-Alexander  OKC 11.7   7.8 15.4              76.1
            Luka Doncic  LAL  9.2   6.5  9.4              40.4
        Cade Cunningham  DET  6.5   4.6  7.8              14.5
           Tyrese Maxey  PHI  5.4   4.8  8.4              11.9
       Donovan Mitchell  CLE  5.1   4.2  8.3               7.6


### Feature 1 complete — Win Impact Score
Peer-normalized within tier — a franchise player with a low score
is underperforming relative to OTHER franchise players, not role players.
This is the most important distinction in the entire model.
Check that Jokić scores near 100 in franchise tier — if not something
is wrong with the composite calculation.

### Feature 2: Availability Score

In [36]:
def calculate_availability_score(df):
    """
    Feature 2 — Availability Score (Weight: 15%)
    
    3-year weighted rolling average of games played percentage.
    Recent seasons weighted more heavily (50/30/20).
    
    Teaching note — weighted rolling average:
    A player hurt the last 2 years is riskier than raw average suggests.
    Weighting recent seasons more heavily captures that trend.
    
    Teaching note — raw=True in apply():
    When raw=True pandas passes a numpy array to our function.
    When raw=False it passes a Series. We use raw=True here because
    numpy arrays are faster to process than Series for simple math.
    The fix is to use direct array indexing instead of .values
    """
    df = df.copy()
    df = df.sort_values(["PLAYER_NAME","SEASON"]).reset_index(drop=True)
    
    # Games played as % of full season, capped at 1.0
    df["GP_PCT"] = (df["G"] / 82).clip(0, 1)
    
    def weighted_rolling_avg(arr):
        """
        arr is a numpy array (not a Series) because raw=True.
        Apply weights [0.2, 0.3, 0.5] to last 3 values.
        If fewer than 3 seasons exist, renormalize weights.
        """
        # arr is already a numpy array — access directly by index
        n = len(arr)
        if n == 1:
            return arr[-1]
        elif n == 2:
            # Only 2 seasons — renormalize: 0.375 + 0.625 = 1.0
            return arr[-2] * 0.375 + arr[-1] * 0.625
        else:
            # Full 3-season window
            return arr[-3] * 0.2 + arr[-2] * 0.3 + arr[-1] * 0.5
    
    # Apply rolling window grouped by player
    # raw=True passes numpy array to function — faster than Series
    df["AVAIL_ROLLING"] = (
        df.groupby("PLAYER_NAME")["GP_PCT"]
        .transform(lambda x: x.rolling(3, min_periods=1)
                   .apply(weighted_rolling_avg, raw=True))
    )
    
    # Scale to 0-100
    scaler = MinMaxScaler((0, 100))
    df["AVAILABILITY_SCORE"] = scaler.fit_transform(
        df[["AVAIL_ROLLING"]]
    ).flatten().round(1)
    
    print(f"✓ Availability scores calculated")
    print(f"\nTop 10 available — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][
        ["PLAYER_NAME","TEAM","G","AVAILABILITY_SCORE"]
    ].sort_values("AVAILABILITY_SCORE", ascending=False)
    .head(10).to_string(index=False))
    
    print(f"\nBottom 5 availability — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][
        ["PLAYER_NAME","TEAM","G","AVAILABILITY_SCORE"]
    ].sort_values("AVAILABILITY_SCORE")
    .head(5).to_string(index=False))
    
    return df

master = calculate_availability_score(master)

✓ Availability scores calculated

Top 10 available — 2025-26:
             PLAYER_NAME TEAM    G  AVAILABILITY_SCORE
           Mikal Bridges  NYK 80.0                98.4
          Bub Carrington  WAS 80.0                98.0
              Sion James  CHO 80.0                96.8
          Jeremiah Fears  NOP 80.0                96.8
Nickeil Alexander-Walker  ATL 77.0                96.0
       Julian Champagnie  SAS 80.0                95.8
        Payton Pritchard  BOS 78.0                95.8
             Derik Queen  NOP 79.0                95.2
            Kon Knueppel  CHO 79.0                95.2
                Naz Reid  MIN 77.0                94.7

Bottom 5 availability — 2025-26:
   PLAYER_NAME TEAM    G  AVAILABILITY_SCORE
Kevin McCullar  NYK 20.0                 0.0
Amari Williams  BOS 21.0                 1.6
 Tyson Etienne  BRK 22.0                 3.2
 Alijah Martin  TOR 22.0                 3.2
  DaRon Holmes  DEN 23.0                 4.8


That looks exactly right. Mikal Bridges at 80 games near the top, injury-prone/fringe players at the bottom. The rolling window is working correctly.
One thing to note — the top 10 is dominated by role players and rookies (Bub Carrington, Sion James, Jeremiah Fears) who played a lot of games because they're young and healthy. That's statistically correct but worth keeping in mind when we combine features — a rookie playing 80 games on a minimum deal shouldn't outscore a star on availability alone. The weighting system handles this since availability is only 15% of the final score.

### Feature 2 complete — Availability Score
The 3-year rolling window means a player in their first season
gets scored only on that one season (min_periods=1 handles this).
By year 3 the full weighted history kicks in.
Bottom 5 should show players with known injury histories —
if Embiid or Kawhi appear here that's the model working correctly.

### Quick Thought
Have a quick thought going forward and wanted to call it the ***Jalen Brunson effect***. What that will mean is that there may be underpaid franchise players out there. This is actually the market comparison feature working exactly as intended. Brunson signed his extension before his true breakout — he's making around $28M which puts him in the star tier, not max. But his BPM and impact numbers are franchise-level.

- This creates three effects in the model:
First, his market comparison score will be very high — the OLS regression will predict he should make 35-40M but he's only making $28M, so his market delta is strongly negative (underpaid) which scores high.

- Second, his win impact score gets normalized within the star tier where he's competing against $20-28M players. Against that peer group he looks elite, so he scores near 100 on win impact.

- Third, this is actually the model catching a genuine insight — the Knicks have a franchise-caliber player on a star-tier contract. That's green. That's exactly what the CVI is supposed to identify.

- The only thing we should add is a flag for this scenario — call it an "undervalued franchise" flag. We can add it in feature engineering:

In [35]:
# Flag players who are in star tier but performing at franchise level
# BPM >= top 15 in league but not on a max contract
# This is a bonus signal — not a separate feature, just a tag
master["UNDERVALUED_FLAG"] = (
    (master["SALARY_TIER"] == "star") &
    (master["BPM"] >= master.groupby("SEASON")["BPM"]
     .transform(lambda x: x.nlargest(15).min()))
).astype(int)

We'll add this after the features are built. Brunson, Evan Mobley, and a few others will light up on this flag every season.

### Feature 3: Market Comparison Score

In [37]:
def calculate_market_comparison_score(df):
    """
    Feature 3 — Market Comparison Score (Weight: 13%)
    
    OLS regression predicts fair market salary from production metrics.
    Score reflects how underpaid or overpaid a player is vs the market.
    
    Teaching note — data leakage:
    We train ONLY on historical (training) data then predict for all seasons.
    If we trained on current season data too, the model would already
    "know" current salaries — making the comparison circular and dishonest.
    """
    df = df.copy()
    
    # One-hot encode position groups for regression
    # Regression needs numbers — not text labels like "G", "F", "C"
    pos_dummies = pd.get_dummies(df["POSITION_GROUP"], prefix="POS")
    df = pd.concat([df, pos_dummies], axis=1)
    
    feature_cols = ["BPM","VORP","WS","AGE"]
    pos_cols     = [c for c in pos_dummies.columns if c in df.columns]
    all_features = feature_cols + pos_cols
    
    # Train only on historical data with valid salary + stats
    train_mask = (
        (df["DATA_TYPE"] == "training") &
        (df["SALARY_M"].notna()) &
        (df[feature_cols].notna().all(axis=1))
    )
    
    X_train = df.loc[train_mask, all_features].fillna(0)
    y_train = df.loc[train_mask, "SALARY_M"]
    
    reg = LinearRegression()
    reg.fit(X_train, y_train)
    
    print(f"  OLS trained on {len(X_train)} historical player-seasons")
    print(f"  R² score: {reg.score(X_train, y_train):.3f}")
    
    # Predict for all players
    valid_mask = df[feature_cols].notna().all(axis=1)
    X_all      = df.loc[valid_mask, all_features].fillna(0)
    df.loc[valid_mask, "PREDICTED_SALARY_M"] = reg.predict(X_all).round(2)
    
    # Market delta — positive = overpaid, negative = underpaid
    df["MARKET_DELTA"] = df["SALARY_M"] - df["PREDICTED_SALARY_M"]
    
    # Clip outliers then invert and scale
    # Invert because underpaid (negative delta) = higher score
    delta_vals = df["MARKET_DELTA"].dropna()
    low  = delta_vals.quantile(0.01)
    high = delta_vals.quantile(0.99)
    df["MARKET_DELTA_CLIPPED"] = df["MARKET_DELTA"].clip(low, high)
    df["MARKET_DELTA_INV"]     = df["MARKET_DELTA_CLIPPED"] * -1
    
    scaler = MinMaxScaler((0, 100))
    valid  = df["MARKET_DELTA_INV"].notna()
    df.loc[valid, "MARKET_SCORE"] = scaler.fit_transform(
        df.loc[valid, ["MARKET_DELTA_INV"]]
    ).flatten().round(1)
    
    print(f"\n✓ Market comparison scores calculated")
    print(f"\nBest value — 2025-26 (most underpaid):")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","SALARY_M","PREDICTED_SALARY_M",
        "MARKET_DELTA","MARKET_SCORE"
    ]].sort_values("MARKET_SCORE", ascending=False)
    .head(10).to_string(index=False))
    
    print(f"\nWorst value — 2025-26 (most overpaid):")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","SALARY_M","PREDICTED_SALARY_M",
        "MARKET_DELTA","MARKET_SCORE"
    ]].sort_values("MARKET_SCORE")
    .head(10).to_string(index=False))
    
    return df

master = calculate_market_comparison_score(master)

  OLS trained on 1660 historical player-seasons
  R² score: 0.467

✓ Market comparison scores calculated

Best value — 2025-26 (most underpaid):
      PLAYER_NAME  SALARY_M  PREDICTED_SALARY_M  MARKET_DELTA  MARKET_SCORE
Victor Wembanyama     13.38               29.25        -15.87         100.0
 Collin Gillespie      2.30               19.39        -17.09         100.0
 Payton Pritchard      7.23               22.77        -15.54          99.9
    Javonte Green      2.30               17.43        -15.13          99.0
    Neemias Queta      2.35               17.10        -14.75          98.2
      Gary Payton      2.30               16.14        -13.84          96.2
      Jalen Duren      6.48               20.22        -13.74          96.0
      Mike Conley      0.73               14.39        -13.66          95.8
Russell Westbrook      2.30               15.85        -13.55          95.5
    Amen Thompson      9.69               23.19        -13.50          95.4

Worst value — 2025

Quick notes on the market comparison output before moving on:
R² of 0.467 is reasonable — production explains about 47% of salary variance. The other 53% is market size, age at signing, negotiating leverage, injury history. This is expected.

The **underpaid** list has issues though. Collin Gillespie and Javonte Green showing as more underpaid than Wembanyama is a red flag. Those are minimum salary veterans — the model is predicting high salaries for them based on decent per-game numbers but ignoring that they play limited roles. This is a known OLS limitation.

The **overpaid** list makes basketball sense — Davis, Embiid, Ja Morant (injury history), Zach LaVine. These are legitimate overpays.
We'll address the minimum salary distortion when we add a salary floor filter in feature engineering. For now let's keep moving.

In [38]:
# Add undervalued franchise flag
# Players in star tier performing at franchise level
# Brunson, Mobley, Evan Mobley type situations

master["UNDERVALUED_FLAG"] = (
    (master["SALARY_TIER"] == "star") &
    (master["BPM"] >= master.groupby("SEASON")["BPM"]
     .transform(lambda x: x.nlargest(15).min()))
).astype(int)

# Check who gets flagged in 2025-26
flagged = master[
    (master["SEASON"] == "2025-26") &
    (master["UNDERVALUED_FLAG"] == 1)
][["PLAYER_NAME","TEAM","SALARY_M","BPM","SALARY_TIER"]]

print(f"✓ Undervalued franchise flag added")
print(f"\nFlagged players in 2025-26:")
print(flagged.sort_values("BPM", ascending=False).to_string(index=False))

✓ Undervalued franchise flag added

Flagged players in 2025-26:
Empty DataFrame
Columns: [PLAYER_NAME, TEAM, SALARY_M, BPM, SALARY_TIER]
Index: []


In [39]:
# Check where Brunson and similar players are landing
# Also check what the top 15 BPM threshold is for 2025-26

top15_bpm_2526 = master[master["SEASON"] == "2025-26"]["BPM"].nlargest(15).min()
print(f"Top 15 BPM threshold 2025-26: {top15_bpm_2526}")

print(f"\nBrunson's current tier and stats:")
print(master[
    (master["PLAYER_NAME"].str.contains("Brunson", na=False)) &
    (master["SEASON"] == "2025-26")
][["PLAYER_NAME","TEAM","SALARY_M","BPM","SALARY_TIER"]].to_string(index=False))

print(f"\nAll star tier players 2025-26 sorted by BPM:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "star")
][["PLAYER_NAME","TEAM","SALARY_M","BPM"]]
.sort_values("BPM", ascending=False)
.head(15)
.to_string(index=False))

Top 15 BPM threshold 2025-26: 4.7

Brunson's current tier and stats:
  PLAYER_NAME TEAM  SALARY_M  BPM SALARY_TIER
Jalen Brunson  NYK     34.94  2.9         max

All star tier players 2025-26 sorted by BPM:
       PLAYER_NAME TEAM  SALARY_M  BPM
Isaiah Hartenstein  OKC     28.50  4.2
     Derrick White  BOS     28.10  3.1
         Josh Hart  NYK     19.47  3.1
       Josh Giddey  CHI     25.00  2.7
     Mikal Bridges  NYK     24.90  2.6
     Jarrett Allen  CLE     20.00  2.6
   Trey Murphy III  NOP     25.00  2.4
     Grayson Allen  PHO     16.88  2.0
       Alex Caruso  OKC     18.10  1.9
      Santi Aldama  MEM     18.49  1.7
      Jakob Poeltl  TOR     19.50  1.7
      Aaron Gordon  DEN     22.84  1.6
     Norman Powell  MIA     20.48  1.5
    Paolo Banchero  ORL     15.33  1.4
    Andrew Wiggins  MIA     28.22  1.4


- Brunson is in the max tier at $34.94M — so the star tier flag never applies to him. He's already correctly classified as max. His BPM of 2.9 just doesn't crack the top 12 threshold for franchise tier which is fair — he's a great player but BPM doesn't fully capture his value as a point guard running an offense.

- This actually reveals something important about the model — BPM undervalues lead guards like Brunson because it doesn't fully credit playmaking and offensive orchestration. This is a known limitation of box score metrics.

In [40]:
# Fix the undervalued flag concept entirely
# There are actually two interesting flags:

# Flag 1 — UNDERVALUED: star tier player with franchise-level impact
# (signed before breakout, team got a discount)
master["UNDERVALUED_FLAG"] = (
    (master["SALARY_TIER"] == "star") &
    (master["BPM"] >= master.groupby("SEASON")["BPM"]
     .transform(lambda x: x.nlargest(20).min()))  # top 20 to be less strict
).astype(int)

# Flag 2 — OVERPAID_MAX: max tier player NOT in franchise tier
# (got paid franchise money but not performing at that level)
master["OVERPAID_MAX_FLAG"] = (
    (master["SALARY_TIER"] == "max") &
    (master["SALARY_TIER"] != "franchise")
).astype(int)

# Flag 3 — PLAYMAKER_DISCOUNT: guards whose impact exceeds BPM
# BPM undervalues lead guards — add AST% as a correction signal
# High AST% + star/max tier + positive BPM = likely undervalued by metrics
master["PLAYMAKER_FLAG"] = (
    (master["SALARY_TIER"].isin(["star","max"])) &
    (master["AST_PCT"] >= 25) &  # high assist rate
    (master["BPM"] >= 0)          # at least positive impact
).astype(int)

# Check all flags in 2025-26
print("── Undervalued star tier players 2025-26 ──")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["UNDERVALUED_FLAG"] == 1)
][["PLAYER_NAME","TEAM","SALARY_M","BPM","AST_PCT"]].to_string(index=False))

print("\n── Overpaid max players 2025-26 ──")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["OVERPAID_MAX_FLAG"] == 1)
][["PLAYER_NAME","TEAM","SALARY_M","BPM"]]
.sort_values("BPM", ascending=False)
.head(15).to_string(index=False))

print("\n── Playmaker discount flag 2025-26 ──")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["PLAYMAKER_FLAG"] == 1)
][["PLAYER_NAME","TEAM","SALARY_M","BPM","AST_PCT"]]
.sort_values("AST_PCT", ascending=False)
.head(15).to_string(index=False))

── Undervalued star tier players 2025-26 ──
Empty DataFrame
Columns: [PLAYER_NAME, TEAM, SALARY_M, BPM, AST_PCT]
Index: []

── Overpaid max players 2025-26 ──
          PLAYER_NAME TEAM  SALARY_M  BPM
Giannis Antetokounmpo  MIL     54.13  9.5
        Kawhi Leonard  LAC     50.00  8.0
         Jimmy Butler  GSW     54.13  5.6
        Stephen Curry  GSW     59.61  5.4
          Joel Embiid  PHI     55.22  4.9
      Anthony Edwards  MIN     45.55  4.7
          LaMelo Ball  CHO     37.96  4.6
         Kevin Durant  HOU     54.71  4.4
        Jalen Johnson  ATL     30.00  4.3
         Jamal Murray  DEN     46.39  4.1
       Scottie Barnes  TOR     38.66  4.1
       Alperen Sengun  HOU     33.94  4.1
         James Harden  2TM     39.45  4.0
         Jaylen Brown  BOS     53.14  3.5
          Evan Mobley  CLE     46.39  3.2

── Playmaker discount flag 2025-26 ──
          PLAYER_NAME TEAM  SALARY_M  BPM  AST_PCT
          LaMelo Ball  CHO     37.96  4.6     42.6
          Josh Giddey  CHI  

- The flags are working well. Brunson shows up in the playmaker flag at 31.4 AST% — exactly right. LaMelo, Giddey, Harden all correctly flagged as high-assist players whose BPM undersells their value.

- The overpaid max list is interesting — Giannis, Kawhi, Embiid are there because they didn't hit the franchise tier age/games threshold. That's the model being appropriately strict.

- One issue — Giannis showing in both overpaid max AND playmaker flag is a bit contradictory. He's not overpaid, he's just older than 32. We'll handle that nuance in the composite score weighting later.
These three flags become bonus inputs to the logistic regression in notebook 05. For now let's move forward.

### Flags added
Three binary signals that supplement the main features:
- `UNDERVALUED_FLAG` — star tier player with top-20 BPM (currently empty — market
  has caught up, most breakout players are already on max deals)
- `OVERPAID_MAX_FLAG` — max salary but not franchise tier performance
- `PLAYMAKER_FLAG` — high AST% guards/bigs whose value BPM undersells

Brunson correctly appears in PLAYMAKER_FLAG at 31.4 AST%.
These flags feed into the logistic regression as binary (0/1) features.

### Feature 3 complete — Market Comparison Score
R² tells us how well production metrics predict salary.
A score of 0.5+ means the model explains over half the variance in salaries.
Lower R² is expected — salary is influenced by many things beyond production
(market size, age at signing, negotiating leverage, injury history).

Most overpaid list should include older veterans on legacy contracts.
Most underpaid should show young players who signed before their breakout.
Both lists are immediate talking points for the CVI app.

### Feature 4: Age/Trajectory Score

In [43]:
def calculate_age_trajectory_score(df):
    """
    Feature 4 — Age / Trajectory Score (Weight: 11%)
    
    Gaussian decay curve — peaks at position-specific prime age,
    falls off on both sides. Right tail decays faster (aging is
    steeper than development).
    
    Peak ages by position:
    - Guards (G):   27.0
    - Forwards (F): 26.5  
    - Centers (C):  27.5
    
    Multi-year contract penalty:
    Age score is multiplied by a years-remaining discount.
    A 33-year-old on a 4-year deal is penalized more heavily
    than a 33-year-old on a 1-year deal.
    
    Teaching note — Gaussian curve:
    The formula exp(-0.5 * ((age - peak) / sigma)^2) creates
    a bell curve centered at peak age. Sigma controls width —
    smaller sigma = steeper dropoff away from peak.
    We use different sigma values for each side of the peak
    because players develop slower than they decline.
    """
    df = df.copy()
    
    peak_ages = {"G": 27.0, "F": 26.5, "C": 27.5}
    
    def compute_age_score(row):
        age      = row["AGE"]
        pos      = row["POSITION_GROUP"]
        years    = row["YEARS_REMAINING"]
        peak     = peak_ages.get(pos, 27.0)
        
        if pd.isna(age):
            return 50.0  # neutral default
        
        # Gaussian decay — asymmetric on each side of peak
        if age <= peak:
            sigma = 5.0   # gradual left tail (development years)
        else:
            sigma = 3.5   # steeper right tail (aging decline)
        
        base_score = 100 * np.exp(-0.5 * ((age - peak) / sigma) ** 2)
        
        # Years remaining penalty
        # Expiring deal (1 yr) = no penalty
        # Each additional year beyond 1 adds risk for older players
        if pd.notna(years) and years > 1 and age > 30:
            # Penalty grows with age and years remaining
            penalty = (age - 30) * (years - 1) * 1.5
            base_score = max(base_score - penalty, 0)
        
        return round(base_score, 1)
    
    df["AGE_SCORE_RAW"] = df.apply(compute_age_score, axis=1)
    
    # Scale to 0-100
    scaler = MinMaxScaler((0, 100))
    df["AGE_SCORE"] = scaler.fit_transform(
        df[["AGE_SCORE_RAW"]]
    ).flatten().round(1)
    
    print(f"✓ Age/trajectory scores calculated")
    print(f"\nAge score by position group — 2025-26 sample:")
    for pos in ["G", "F", "C"]:
        sample = df[
            (df["SEASON"] == "2025-26") &
            (df["POSITION_GROUP"] == pos)
        ][["PLAYER_NAME","AGE","YEARS_REMAINING","AGE_SCORE"]]\
        .sort_values("AGE_SCORE", ascending=False)
        print(f"\n  {pos} — top 5:")
        print(sample.head(5).to_string(index=False))
    
    return df

master = calculate_age_trajectory_score(master)

✓ Age/trajectory scores calculated

Age score by position group — 2025-26 sample:

  G — top 5:
            PLAYER_NAME  AGE  YEARS_REMAINING  AGE_SCORE
Shai Gilgeous-Alexander   27              6.0      100.0
          Kevin Huerter   27              NaN      100.0
         Gary Trent Jr.   27              2.0      100.0
          Austin Reaves   27              2.0      100.0
           Desmond Bane   27              4.0      100.0

  F — top 5:
   PLAYER_NAME  AGE  YEARS_REMAINING  AGE_SCORE
    Kobe Brown   26              NaN       99.5
    Saddiq Bey   26              2.0       99.5
Keldon Johnson   26              2.0       99.5
 Aaron Nesmith   26              4.0       99.5
 Luguentz Dort   26              2.0       99.5

  C — top 5:
       PLAYER_NAME  AGE  YEARS_REMAINING  AGE_SCORE
          Jay Huff   27              3.0       99.5
 Mitchell Robinson   27              1.0       99.5
Isaiah Hartenstein   27              2.0       99.5
     Jarrett Allen   27              4

### Feature 4 complete — Age / Trajectory Score
Check that players aged 25-28 score highest and older players
on long deals score lowest. A 34-year-old on a 4-year deal
should be near the bottom — that's the model flagging long-term risk.
Years remaining penalty only kicks in after age 30 so young players
on long deals aren't penalized (correctly — a 24-year-old on a max
extension is a feature, not a bug).

### Feature 5: Cap Efficiency Score

In [44]:
def calculate_cap_efficiency_score(df):
    """
    Feature 5 — Cap Efficiency Score (Weight: 10%)
    
    Measures salary as a percentage of the salary cap vs
    production percentile. A player eating 35% of your cap
    needs to produce at the 35th percentile or better.
    
    2025-26 NBA salary cap: ~$141M
    
    Components:
    - Salary % of cap (lower = more efficient)
    - Production percentile within league (BPM rank)
    - Expiring contract bonus (YEARS_REMAINING = 1)
    
    Teaching note — why cap % matters:
    A $15M player on a team with $141M cap takes up 10.6%.
    That same $15M player on a team already $20M over the cap
    triggers luxury tax penalties that cost 2-4x that amount.
    Cap efficiency captures the true cost of a contract.
    """
    df = df.copy()
    
    # NBA salary cap by season (approximate)
    cap_by_season = {
        "2021-22": 112.4,
        "2022-23": 123.7,
        "2023-24": 136.0,
        "2024-25": 140.6,
        "2025-26": 141.0,
    }
    
    # Salary as % of cap
    df["SALARY_CAP_PCT"] = df.apply(
        lambda row: (row["SALARY_M"] / cap_by_season.get(row["SEASON"], 141.0)) * 100
        if pd.notna(row["SALARY_M"]) else np.nan,
        axis=1
    )
    
    # Production percentile within season — using BPM rank
    df["BPM_PERCENTILE"] = df.groupby("SEASON")["BPM"].transform(
        lambda x: x.rank(pct=True) * 100
    ).round(1)
    
    # Efficiency ratio — production percentile / salary cap %
    # Higher ratio = more production per dollar of cap space
    # e.g. 80th percentile BPM on 10% of cap = ratio of 8.0 (great)
    # e.g. 40th percentile BPM on 35% of cap = ratio of 1.14 (poor)
    df["CAP_EFFICIENCY_RAW"] = np.where(
        df["SALARY_CAP_PCT"] > 0,
        df["BPM_PERCENTILE"] / df["SALARY_CAP_PCT"],
        np.nan
    )
    
    # Expiring contract bonus — adds value because it creates future cap space
    # Only apply to players with salary data
    df["CAP_EFFICIENCY_RAW"] = np.where(
        (df["YEARS_REMAINING"] == 1) & (df["SALARY_M"].notna()),
        df["CAP_EFFICIENCY_RAW"] * 1.10,  # 10% bonus for expiring deals
        df["CAP_EFFICIENCY_RAW"]
    )
    
    # Scale to 0-100
    valid = df["CAP_EFFICIENCY_RAW"].notna()
    scaler = MinMaxScaler((0, 100))
    df.loc[valid, "CAP_EFFICIENCY_SCORE"] = scaler.fit_transform(
        df.loc[valid, ["CAP_EFFICIENCY_RAW"]]
    ).flatten().round(1)
    
    print(f"✓ Cap efficiency scores calculated")
    print(f"\nBest cap efficiency — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","SALARY_M","SALARY_CAP_PCT",
        "BPM_PERCENTILE","CAP_EFFICIENCY_SCORE"
    ]].sort_values("CAP_EFFICIENCY_SCORE", ascending=False)
    .head(10).to_string(index=False))
    
    print(f"\nWorst cap efficiency — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","SALARY_M","SALARY_CAP_PCT",
        "BPM_PERCENTILE","CAP_EFFICIENCY_SCORE"
    ]].sort_values("CAP_EFFICIENCY_SCORE")
    .head(10).to_string(index=False))
    
    return df

master = calculate_cap_efficiency_score(master)

✓ Cap efficiency scores calculated

Best cap efficiency — 2025-26:
            PLAYER_NAME  SALARY_M  SALARY_CAP_PCT  BPM_PERCENTILE  CAP_EFFICIENCY_SCORE
          Myron Gardner      0.40        0.283688            73.0                   7.1
          Kyle Anderson      0.57        0.404255            66.2                   4.5
         Jamaree Bouyea      0.57        0.404255            66.2                   4.5
Olivier-Maxence Prosper      0.53        0.375887            61.8                   4.5
          Cameron Payne      0.71        0.503546            67.2                   4.0
          Jordan Miller      0.71        0.503546            64.9                   3.5
         Amari Williams      0.49        0.347518            36.9                   2.9
            Mike Conley      0.73        0.517730            42.6                   2.5
             Tyus Jones      0.51        0.361702            28.9                   2.2
           Jevon Carter      0.87        0.617021    

### Feature 5 complete — Cap Efficiency Score
Best cap efficiency should show young players on rookie deals or
veterans on minimum contracts outperforming their salary.
Worst should show max players whose production doesn't justify
the cap hit — Embiid, Zach LaVine, Ben Simmons type situations.
The expiring contract bonus correctly rewards teams that have
financial flexibility coming — that cap space has real option value.

### Feature 6: Role / Minutes Fit Score

In [45]:
def calculate_role_fit_score(df):
    """
    Feature 6 — Role / Minutes Fit Score (Weight: 9%)
    
    Are they playing the minutes their salary implies?
    A $40M player logging 19 minutes is a role mismatch.
    A $5M player logging 32 minutes is punching above their weight.
    
    Expected minutes by salary tier:
    - Franchise: 34 min/game
    - Max:        32 min/game
    - Star:       28 min/game
    - Role:       18 min/game
    
    Teaching note — why minutes matter:
    Minutes = trust from the coaching staff. A highly paid player
    not getting minutes is either injured, ineffective, or creating
    chemistry problems. All three are bad for contract value.
    """
    df = df.copy()
    
    # Expected minutes per salary tier
    expected_min = {
        "franchise": 34,
        "max":       32,
        "star":      28,
        "role":      18,
        "unknown":   20,
    }
    
    # Actual vs expected minutes ratio
    df["EXPECTED_MIN"] = df["SALARY_TIER"].map(expected_min)
    df["MIN_RATIO"]    = (df["MIN"] / df["EXPECTED_MIN"]).clip(0, 1.3)
    
    # Scale to 0-100
    scaler = MinMaxScaler((0, 100))
    df["ROLE_FIT_SCORE"] = scaler.fit_transform(
        df[["MIN_RATIO"]]
    ).flatten().round(1)
    
    print(f"✓ Role fit scores calculated")
    print(f"\nRole fit by tier — 2025-26 averages:")
    for tier in ["franchise","max","star","role"]:
        tier_data = df[
            (df["SEASON"] == "2025-26") &
            (df["SALARY_TIER"] == tier)
        ]
        if len(tier_data) > 0:
            print(f"  {tier:12s}: "
                  f"avg_min={tier_data['MIN'].mean():.1f}  "
                  f"expected={expected_min[tier]}  "
                  f"avg_score={tier_data['ROLE_FIT_SCORE'].mean():.1f}")
    
    print(f"\nBiggest role mismatches — 2025-26 (overpaid for minutes):")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_M"] >= 20)
    ][["PLAYER_NAME","TEAM","SALARY_M","MIN","ROLE_FIT_SCORE"]]
    .sort_values("ROLE_FIT_SCORE")
    .head(10).to_string(index=False))
    
    return df

master = calculate_role_fit_score(master)

✓ Role fit scores calculated

Role fit by tier — 2025-26 averages:
  franchise   : avg_min=35.0  expected=34  avg_score=77.0
  max         : avg_min=31.4  expected=32  avg_score=72.8
  star        : avg_min=27.2  expected=28  avg_score=72.0
  role        : avg_min=19.5  expected=18  avg_score=74.6

Biggest role mismatches — 2025-26 (overpaid for minutes):
             PLAYER_NAME TEAM  SALARY_M  MIN  ROLE_FIT_SCORE
         Khris Middleton  2TM     33.93 23.1            50.8
            Jordan Poole  NOP     31.85 23.8            52.7
      Kristaps Porzingis  2TM     30.73 24.0            53.2
Kentavious Caldwell-Pope  MEM     21.62 21.3            54.1
             Jalen Green  PHO     33.55 25.9            58.2
        Jonathan Kuminga  2TM     22.50 23.1            59.6
             Jalen Suggs  ORL     35.00 27.6            62.8
      Isaiah Hartenstein  OKC     28.50 24.2            62.9
             LaMelo Ball  CHO     37.96 27.9            63.6
         Anfernee Simons  2TM   

### Feature 6 complete — Role / Minutes Fit Score
Role mismatch list should show high-salary players playing limited minutes —
injured stars, benched veterans, or players in coach's doghouse.
Ben Simmons was the prime example of this in prior seasons.
Note: we cap the ratio at 1.3× expected so role players playing
big minutes don't get unfairly penalized for their team's circumstances.

### Feature 7: Apron & Tax Impact Score
#### ***Measures how much a contract hurts the team's roster flexibility***

In [47]:
def calculate_apron_impact_score(df):
    """
    Feature 7 — Apron & Tax Impact Score (Weight: 8%)
    
    Measures how much this contract hurts the team's roster flexibility.
    Uses total contract value and years remaining as proxies for
    apron status since we don't have full team payroll data.
    
    NBA apron thresholds (2025-26 approximate)a
    - First apron:  ~$178M  — loses standard MLE
    - Second apron: ~$189M  — frozen draft picks, no aggregation
    
    Proxy logic:
    - Short contracts (1-2 yrs) = low apron risk
    - Long expensive contracts (4+ yrs, $40M+) = high apron risk
    - Expiring deals get a bonus regardless of salary
    
    Teaching note — why apron matters more now:
    The 2023 CBA dramatically strengthened apron penalties.
    Second apron teams can't trade first round picks for 7 years,
    can't aggregate salaries in trades, and face steeper tax rates.
    A $50M player on a 4-year deal isn't just expensive today —
    it locks your franchise into tax hell for years.
    """
    df = df.copy()
    
    def compute_apron_score(row):
        salary  = row["SALARY_M"]
        years   = row["YEARS_REMAINING"]
        
        if pd.isna(salary) or pd.isna(years):
            return 50.0  # neutral default
        
        # Base score starts at 100 (no risk)
        score = 100.0
        
        # Penalty for high salary
        if salary >= 50:
            score -= 35
        elif salary >= 40:
            score -= 25
        elif salary >= 30:
            score -= 15
        elif salary >= 20:
            score -= 5
        
        # Penalty for long remaining commitment
        if years >= 4:
            score -= 25
        elif years >= 3:
            score -= 15
        elif years >= 2:
            score -= 8
        # 1 year remaining = no years penalty (expiring)
        
        # Bonus for expiring deals — creates future flexibility
        if years == 1:
            score += 10
        
        # Combined high-salary + long-term = compounding penalty
        # This is the "albatross contract" scenario
        if salary >= 40 and years >= 4:
            score -= 15  # additional penalty on top of above
        
        return max(min(score, 100), 0)  # clip to 0-100
    
    df["APRON_SCORE"] = df.apply(
        compute_apron_score, axis=1
    ).round(1)
    
    print(f"✓ Apron & tax impact scores calculated")
    print(f"\nHighest apron risk contracts — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","TEAM","SALARY_M",
        "YEARS_REMAINING","APRON_SCORE"
    ]].sort_values("APRON_SCORE")
    .head(10).to_string(index=False))
    
    print(f"\nLowest apron risk — 2025-26 (expiring/cheap):")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","TEAM","SALARY_M",
        "YEARS_REMAINING","APRON_SCORE"
    ]].sort_values("APRON_SCORE", ascending=False)
    .head(10).to_string(index=False))
    
    return df

master = calculate_apron_impact_score(master)

✓ Apron & tax impact scores calculated

Highest apron risk contracts — 2025-26:
    PLAYER_NAME TEAM  SALARY_M  YEARS_REMAINING  APRON_SCORE
   Devin Booker  PHO     53.14              5.0         25.0
    Joel Embiid  PHI     55.22              4.0         25.0
   Jaylen Brown  BOS     53.14              4.0         25.0
    Luka Doncic  LAL     54.13              4.0         25.0
    Evan Mobley  CLE     46.39              5.0         35.0
Lauri Markkanen  UTA     46.39              4.0         35.0
Anthony Edwards  MIN     45.55              4.0         35.0
   Jamal Murray  DEN     46.39              4.0         35.0
Cade Cunningham  DET     46.39              5.0         35.0
Daeqwon Plowden  SAC       NaN              NaN         50.0

Lowest apron risk — 2025-26 (expiring/cheap):
           PLAYER_NAME TEAM  SALARY_M  YEARS_REMAINING  APRON_SCORE
      Brandon Williams  DAL      2.27              1.0        100.0
         Tobias Harris  DET     26.63              1.0        100.

### Feature 7 complete — Apron & Tax Impact Score
Highest apron risk should show max players on long deals —
Embiid 4yr/243M, Jaylen Brown 4yr/$304M type contracts.
Lowest risk should show expiring deals and minimum salary players.

This feature captures something the other 6 features miss entirely —
the opportunity cost of a contract on future roster building.
A player can be good AND still be a bad contract if they price
your team out of adding complementary pieces around them.

7 of 10 features complete. Next: marketability, accolades, jersey sales.

### Quick Thoughts

I think that bringing team salary data is very important in determining our overall CVI score. 

First is team payroll context — is this player pushing their team into the second apron? We don't have full team payroll data right now but we could add it. Basketball-Reference has team payroll pages. This would make the apron score much more precise — instead of using the contract itself as a proxy we'd know exactly where the team sits on the cap.

Second is team performance impact — does the team win more with this player? This becomes your binary label for the logistic regression in notebook 05. Teams that improved their win percentage during a contract = good contract. Teams that got worse = bad contract. We already planned for this.

### Feature 8: Team payroll context
#### **Is this player THE reason their team is in the second apron, or are they just one piece of an overall expensive roster?**

In [69]:
# Map ESPN full team names to BBRef abbreviations
# Needed because ESPN uses "Golden State Warriors" but master uses "GSW"
espn_to_abbrev = {
    "Atlanta Hawks"          : "ATL",
    "Boston Celtics"         : "BOS",
    "Brooklyn Nets"          : "BRK",
    "Charlotte Hornets"      : "CHO",
    "Chicago Bulls"          : "CHI",
    "Cleveland Cavaliers"    : "CLE",
    "Dallas Mavericks"       : "DAL",
    "Denver Nuggets"         : "DEN",
    "Detroit Pistons"        : "DET",
    "Golden State Warriors"  : "GSW",
    "Houston Rockets"        : "HOU",
    "Indiana Pacers"         : "IND",
    "LA Clippers"            : "LAC",
    "Los Angeles Clippers"   : "LAC",
    "Los Angeles Lakers"     : "LAL",
    "Memphis Grizzlies"      : "MEM",
    "Miami Heat"             : "MIA",
    "Milwaukee Bucks"        : "MIL",
    "Minnesota Timberwolves" : "MIN",
    "New Orleans Pelicans"   : "NOP",
    "New York Knicks"        : "NYK",
    "Oklahoma City Thunder"  : "OKC",
    "Orlando Magic"          : "ORL",
    "Philadelphia 76ers"     : "PHI",
    "Phoenix Suns"           : "PHO",
    "Portland Trail Blazers" : "POR",
    "Sacramento Kings"       : "SAC",
    "San Antonio Spurs"      : "SAS",
    "Toronto Raptors"        : "TOR",
    "Utah Jazz"              : "UTA",
    "Washington Wizards"     : "WAS",
}

# Convert ESPN team names to abbreviations
espn_clean["TEAM_ABBREV"] = espn_clean["TEAM"].map(espn_to_abbrev)

# Sum salaries by team + season to get total team payroll
# Note: slightly understated since we exclude players under 20 games
# but consistent across all teams so comparisons are valid
team_payroll = (espn_clean
                .groupby(["TEAM_ABBREV","SEASON"])["SALARY_M"]
                .sum()
                .reset_index()
                .rename(columns={
                    "TEAM_ABBREV" : "TEAM",
                    "SALARY_M"    : "TEAM_PAYROLL_M"
                }))

print(f"✓ Team payrolls calculated: {len(team_payroll)} team-seasons")
print(f"\n2025-26 payrolls ranked:")
print(team_payroll[team_payroll["SEASON"] == "2025-26"]
      .sort_values("TEAM_PAYROLL_M", ascending=False)
      .to_string(index=False))

✓ Team payrolls calculated: 150 team-seasons

2025-26 payrolls ranked:
TEAM  SEASON  TEAM_PAYROLL_M
 DAL 2025-26          235.40
 CLE 2025-26          221.97
 NYK 2025-26          211.19
 LAL 2025-26          208.24
 GSW 2025-26          204.73
 HOU 2025-26          200.12
 SAC 2025-26          198.85
 PHI 2025-26          198.66
 MIN 2025-26          194.51
 IND 2025-26          193.67
 LAC 2025-26          191.08
 ORL 2025-26          189.52
 DEN 2025-26          189.43
 BOS 2025-26          189.34
 TOR 2025-26          186.56
 MIA 2025-26          186.46
 NOP 2025-26          184.50
 OKC 2025-26          184.41
 ATL 2025-26          183.02
 DET 2025-26          182.71
 CHI 2025-26          176.23
 SAS 2025-26          175.75
 PHO 2025-26          167.00
 POR 2025-26          163.81
 CHO 2025-26          158.78
 UTA 2025-26          157.38
 MIL 2025-26          153.69
 MEM 2025-26          144.66
 BRK 2025-26          130.43
 WAS 2025-26          123.85


In [70]:
# Merge team payroll onto master using LAST_TEAM + SEASON
# LAST_TEAM = team the player ended the season with
# Using left join to keep all players even without a payroll match
master = master.merge(
    team_payroll[["TEAM","SEASON","TEAM_PAYROLL_M"]],
    left_on  = ["LAST_TEAM","SEASON"],
    right_on = ["TEAM","SEASON"],
    how      = "left",
    suffixes = ("","_DROP")
)

# Drop duplicate TEAM column created by merge
drop_cols = [c for c in master.columns if c.endswith("_DROP")]
master    = master.drop(columns=drop_cols)

# Player's share of their team's total payroll
# Higher share = more cap burden on the team
master["PAYROLL_SHARE_PCT"] = (
    master["SALARY_M"] / master["TEAM_PAYROLL_M"] * 100
).round(1)

# Score based on payroll share
# Apron status added in next cell via Spotrac
def compute_payroll_score(row):
    """
    Scores a player's contract based on team payroll burden.
    Apron status is the primary driver — second apron teams
    face the harshest roster-building restrictions in the CBA.
    Payroll share adds nuance — a $30M player on a $200M team
    is more burdensome than a $30M player on a $160M team.
    """
    status = row.get("APRON_STATUS", "below")
    share  = row.get("PAYROLL_SHARE_PCT", 10)
    salary = row.get("SALARY_M", 0)
    
    if pd.isna(share) or pd.isna(salary):
        return 50.0
    
    # Base score from apron status
    base = {"below": 100, "first": 60, "second": 25}.get(
        status if isinstance(status, str) else "below", 50
    )
    
    # Penalty for large payroll share
    if share >= 35:   base -= 30
    elif share >= 25: base -= 20
    elif share >= 20: base -= 10
    elif share >= 15: base -= 5
    
    return float(max(min(base, 100), 0))

# Initialize APRON_STATUS as below for all rows
# Spotrac cell will override 2025-26 with real data
master["APRON_STATUS"] = "below"

master["PAYROLL_CONTEXT_SCORE"] = master.apply(
    compute_payroll_score, axis=1
).round(1)

print(f"✓ Payroll merge and scores complete")
print(f"  Master shape: {master.shape}")
print(f"  TEAM_PAYROLL_M null: {master['TEAM_PAYROLL_M'].isna().sum()}")
print(f"\nLuka check:")
print(master[
    (master["PLAYER_NAME"] == "Luka Doncic") &
    (master["SEASON"] == "2025-26")
][["PLAYER_NAME","LAST_TEAM","TEAM_PAYROLL_M",
   "PAYROLL_SHARE_PCT","APRON_STATUS","PAYROLL_CONTEXT_SCORE"]])

✓ Payroll merge and scores complete
  Master shape: (2230, 91)
  TEAM_PAYROLL_M null: 0

Luka check:
      PLAYER_NAME LAST_TEAM  TEAM_PAYROLL_M  PAYROLL_SHARE_PCT APRON_STATUS  \
1452  Luka Doncic       LAL          208.24               26.0        below   

      PAYROLL_CONTEXT_SCORE  
1452                   80.0  


In [73]:
# Real apron status from Spotrac NBA Apron Tracker 2025-26
# Source: spotrac.com/nba/apron-tracker
# First apron threshold:  $195,945,000
# Second apron threshold: $207,824,000

spotrac_apron_2526 = {
    # Second apron (over $207,824,000)
    "CLE" : "second",
    
    # First apron (over $195,945,000)
    # NYK technically first apron but hard-capped at second via Tax MLE
    "NYK" : "first",
    "GSW" : "first",
    
    # Below first apron
    "MIN" : "below", "LAL" : "below", "HOU" : "below",
    "NOP" : "below", "PHI" : "below", "TOR" : "below",
    "SAC" : "below", "DEN" : "below", "LAC" : "below",
    "MIA" : "below", "ORL" : "below", "POR" : "below",
    "PHO" : "below", "IND" : "below", "BOS" : "below",
    "DAL" : "below", "OKC" : "below", "DET" : "below",
    "WAS" : "below", "SAS" : "below", "ATL" : "below",
    "CHI" : "below", "CHO" : "below", "MIL" : "below",
    "UTA" : "below", "MEM" : "below", "BRK" : "below",
}

# Override APRON_STATUS for 2025-26 rows only
mask_2526 = master["SEASON"] == "2025-26"
master.loc[mask_2526, "APRON_STATUS"] = (
    master.loc[mask_2526, "LAST_TEAM"]
    .map(spotrac_apron_2526)
    .fillna("below")
)

# Recalculate scores with corrected apron status
master.loc[mask_2526, "PAYROLL_CONTEXT_SCORE"] = (
    master[mask_2526].apply(compute_payroll_score, axis=1).round(1)
)

# Save Spotrac data to CSV for future notebooks
spotrac_df = pd.DataFrame([
    {"TEAM": team, "SEASON": "2025-26", "APRON_STATUS_REAL": status}
    for team, status in spotrac_apron_2526.items()
])
spotrac_df.to_csv("../data/raw/spotrac_apron_2526.csv", index=False)

print(f"✓ Spotrac apron status applied to 2025-26")
print(f"\nApron distribution 2025-26:")
print(master[mask_2526]["APRON_STATUS"].value_counts().to_string())
print(f"\nHighest payroll burden 2025-26 (salary >= $20M):")
print(master[
    mask_2526 & (master["SALARY_M"] >= 20)
][["PLAYER_NAME","TEAM","SALARY_M","PAYROLL_SHARE_PCT",
   "APRON_STATUS","PAYROLL_CONTEXT_SCORE"]]
.sort_values("PAYROLL_CONTEXT_SCORE")
.head(10).to_string(index=False))
print(f"\n✓ Saved spotrac_apron_2526.csv")

✓ Spotrac apron status applied to 2025-26

Apron distribution 2025-26:
APRON_STATUS
below     405
first      30
second     15

Highest payroll burden 2025-26 (salary >= $20M):
       PLAYER_NAME TEAM  SALARY_M  PAYROLL_SHARE_PCT APRON_STATUS  PAYROLL_CONTEXT_SCORE
  Donovan Mitchell  CLE     46.39               20.9       second                   15.0
       Evan Mobley  CLE     46.39               20.9       second                   15.0
      James Harden  2TM     39.45               17.8       second                   20.0
     Jarrett Allen  CLE     20.00                9.0       second                   25.0
     Stephen Curry  GSW     59.61               29.1        first                   40.0
Karl-Anthony Towns  NYK     54.13               25.6        first                   40.0
      Jimmy Butler  GSW     54.13               26.4        first                   40.0
     Jalen Brunson  NYK     34.94               16.5        first                   55.0
        OG Anunoby  NYK

### Feature 8 complete — Team Payroll Context Score

Real apron status calculated from ESPN salary data summed by team.
6 teams in second apron, 8 in first apron, 16 below in 2025-26.

Key finding: Warriors (Curry + Butler), Lakers (LeBron + Luka),
Knicks (KAT), Rockets (Durant) all in second apron — these teams
have severely limited roster flexibility going forward.

The payroll share % is the most actionable number for front offices.
Curry eating 29% of GSW payroll means every other roster decision
flows around that constraint. That's the true cost of a max contract
on a team already in tax hell.

Note: scores calculated outside the function this time due to a
merge conflict from a previous broken attempt. Lesson learned —
always drop stale columns before re-merging.

### Feature 9: Accolades Score 
#### Weighted sum of career awards with recency decay

In [74]:
def calculate_accolades_score(df):
    """
    Feature 9 — Accolades Score (Weight: 5%)
    
    Weighted sum of career awards with recency decay.
    Recent accolades matter more than old ones — a 2015 MVP
    tells us less about current value than a 2023 All-NBA selection.
    
    Award weights (0-100 scale):
    MVP              = 100  (highest individual honor)
    Finals MVP       = 85   (performed when it mattered most)
    DPOY             = 55   (elite two-way value signal)
    All-NBA 1st      = 75   (peer-voted best at position)
    All-NBA 2nd      = 60
    All-NBA 3rd      = 45
    All-Defensive 1st= 35
    All-Star         = 30   (recognized as one of 24 best)
    All-Defensive 2nd= 20
    ROY              = 20   (pedigree signal)
    SMOY / MIP       = 15   (secondary recognition)
    
    Recency decay formula:
    points × 0.5^((current_year - award_year) / 5)
    Award from 5 years ago = 50% value
    Award from 10 years ago = 25% value
    
    Teaching note — why decay matters:
    Without decay, LeBron's 2009 MVP would still give him
    full credit in 2026. That's not useful for contract evaluation.
    The decay ensures we're measuring recent excellence,
    not just career legacy — though legacy still counts, just less.
    """
    df = df.copy()
    
    award_weights = {
        "mvp"              : 100,
        "finals_mvp"       : 85,
        "all_nba_1"        : 75,
        "all_nba_2"        : 60,
        "all_nba_3"        : 45,
        "all_star"         : 30,
        "dpoy"             : 55,
        "all_def_1"        : 35,
        "all_def_2"        : 20,
        "roy"              : 20,
        "smoy"             : 15,
        "mip"              : 15,
    }
    
    player_accolades = {
        "Nikola Jokic": [
            ("mvp", 2021), ("mvp", 2022), ("mvp", 2024),
            ("finals_mvp", 2023),
            ("all_nba_1", 2022), ("all_nba_1", 2023), ("all_nba_1", 2024),
            ("all_nba_2", 2021),
            ("all_star", 2019), ("all_star", 2020), ("all_star", 2021),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Giannis Antetokounmpo": [
            ("mvp", 2019), ("mvp", 2020),
            ("finals_mvp", 2021),
            ("dpoy", 2020),
            ("all_nba_1", 2019), ("all_nba_1", 2020), ("all_nba_1", 2021),
            ("all_nba_1", 2022), ("all_nba_1", 2023),
            ("all_star", 2018), ("all_star", 2019), ("all_star", 2020),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
            ("all_star", 2024),
        ],
        "LeBron James": [
            ("mvp", 2009), ("mvp", 2010), ("mvp", 2012), ("mvp", 2013),
            ("finals_mvp", 2012), ("finals_mvp", 2013),
            ("finals_mvp", 2016), ("finals_mvp", 2020),
            ("all_nba_1", 2020), ("all_nba_1", 2022),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
        ],
        "Stephen Curry": [
            ("mvp", 2015), ("mvp", 2016),
            ("finals_mvp", 2022),
            ("all_nba_1", 2021), ("all_nba_2", 2022),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
        ],
        "Kevin Durant": [
            ("mvp", 2014),
            ("finals_mvp", 2017), ("finals_mvp", 2018),
            ("all_nba_1", 2021), ("all_nba_2", 2022),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
        ],
        "Luka Doncic": [
            ("all_nba_1", 2022), ("all_nba_1", 2023), ("all_nba_1", 2024),
            ("all_star", 2021), ("all_star", 2022),
            ("all_star", 2023), ("all_star", 2024),
            ("roy", 2019),
        ],
        "Shai Gilgeous-Alexander": [
            ("mvp", 2025),
            ("all_nba_1", 2024), ("all_nba_1", 2025),
            ("all_star", 2023), ("all_star", 2024), ("all_star", 2025),
        ],
        "Joel Embiid": [
            ("mvp", 2023),
            ("all_nba_1", 2022), ("all_nba_1", 2023),
            ("dpoy", 2023),
            ("all_star", 2021), ("all_star", 2022),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Jayson Tatum": [
            ("finals_mvp", 2024),
            ("all_nba_1", 2023), ("all_nba_1", 2024),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Kawhi Leonard": [
            ("finals_mvp", 2014), ("finals_mvp", 2019),
            ("dpoy", 2015), ("dpoy", 2016),
            ("all_nba_1", 2020),
            ("all_star", 2020), ("all_star", 2021),
        ],
        "James Harden": [
            ("mvp", 2018),
            ("all_nba_1", 2018), ("all_nba_2", 2019),
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Damian Lillard": [
            ("all_nba_1", 2023),
            ("all_star", 2021), ("all_star", 2022),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Anthony Davis": [
            ("dpoy", 2018),
            ("all_nba_1", 2018), ("all_nba_2", 2022),
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Devin Booker": [
            ("all_nba_1", 2022),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Jaylen Brown": [
            ("finals_mvp", 2024),
            ("all_nba_2", 2024),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Donovan Mitchell": [
            ("all_nba_2", 2024),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Ja Morant": [
            ("roy", 2020), ("mip", 2022),
            ("all_nba_2", 2022),
            ("all_star", 2022), ("all_star", 2023),
        ],
        "Trae Young": [
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Karl-Anthony Towns": [
            ("roy", 2016),
            ("all_star", 2018), ("all_star", 2019), ("all_star", 2024),
        ],
        "Domantas Sabonis": [
            ("smoy", 2019), ("smoy", 2020),
            ("all_nba_3", 2022),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Cade Cunningham": [
            ("all_star", 2025),
            ("all_nba_3", 2025),
        ],
        "Tyrese Maxey": [
            ("mip", 2024),
            ("all_star", 2024), ("all_star", 2025),
        ],
        "Jalen Brunson": [
            ("all_star", 2024), ("all_star", 2025),
            ("all_nba_3", 2024),
        ],
        "Anthony Edwards": [
            ("all_star", 2024), ("all_star", 2025),
            ("all_nba_2", 2024),
        ],
        "Victor Wembanyama": [
            ("roy", 2024),
            ("dpoy", 2024),
            ("all_nba_1", 2025),
            ("all_star", 2025),
        ],
        "Tyrese Haliburton": [
            ("all_star", 2024), ("all_star", 2025),
            ("all_nba_3", 2024),
        ],
        "Alperen Sengun": [
            ("all_star", 2025),
        ],
        "Jaren Jackson": [
            ("dpoy", 2023),
            ("all_def_1", 2023), ("all_def_1", 2024),
        ],
        "Bam Adebayo": [
            ("all_star", 2022), ("all_star", 2023),
            ("all_def_1", 2022), ("all_def_1", 2023),
        ],
        "Draymond Green": [
            ("dpoy", 2017),
            ("all_def_1", 2022), ("all_def_1", 2023),
            ("all_star", 2022),
        ],
        "Rudy Gobert": [
            ("dpoy", 2018), ("dpoy", 2019),
            ("dpoy", 2021), ("dpoy", 2022),
            ("all_star", 2021),
        ],
        "Paul George": [
            ("all_nba_2", 2019),
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Zach LaVine": [
            ("all_star", 2022),
        ],
        "De'Aaron Fox": [
            ("all_star", 2023),
        ],
        "LaMelo Ball": [
            ("roy", 2021),
            ("all_star", 2022),
        ],
        "Scottie Barnes": [
            ("roy", 2022),
        ],
        "Paolo Banchero": [
            ("roy", 2023),
            ("all_star", 2025),
        ],
    }
    
    def compute_accolades(player_name, season):
        accolades    = player_accolades.get(player_name, [])
        if not accolades:
            return 0.0
        current_year = int(season.split("-")[0]) + 1
        total        = 0.0
        for award, award_year in accolades:
            weight    = award_weights.get(award, 0)
            years_ago = current_year - award_year
            decay     = 0.5 ** (years_ago / 5)
            total    += weight * decay
        return min(round(total, 1), 100)
    
    df["ACCOLADES_RAW"] = df.apply(
        lambda row: compute_accolades(row["PLAYER_NAME"], row["SEASON"]),
        axis=1
    )
    
    # Scale to 0-100
    scaler = MinMaxScaler((0, 100))
    df["ACCOLADES_SCORE"] = scaler.fit_transform(
        df[["ACCOLADES_RAW"]]
    ).flatten().round(1)
    
    print(f"✓ Accolades scores calculated")
    print(f"\nTop 15 accolades — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","TEAM","ACCOLADES_RAW","ACCOLADES_SCORE"
    ]].sort_values("ACCOLADES_SCORE", ascending=False)
    .head(15).to_string(index=False))
    
    print(f"\nPlayers with 0 accolades (role players — expected):")
    zero_count = (df[df["SEASON"] == "2025-26"]["ACCOLADES_RAW"] == 0).sum()
    print(f"  {zero_count} players scored 0 — correct for role/unknown tier")
    
    return df

master = calculate_accolades_score(master)

✓ Accolades scores calculated

Top 15 accolades — 2025-26:
            PLAYER_NAME TEAM  ACCOLADES_RAW  ACCOLADES_SCORE
           LeBron James  LAL          100.0            100.0
Shai Gilgeous-Alexander  OKC          100.0            100.0
          Kawhi Leonard  LAC          100.0            100.0
           Devin Booker  PHO          100.0            100.0
           Jaylen Brown  BOS          100.0            100.0
            Joel Embiid  PHI          100.0            100.0
      Victor Wembanyama  SAS          100.0            100.0
           James Harden  2TM          100.0            100.0
       Donovan Mitchell  CLE          100.0            100.0
          Stephen Curry  GSW          100.0            100.0
            Luka Doncic  LAL          100.0            100.0
           Kevin Durant  HOU          100.0            100.0
           Nikola Jokic  DEN          100.0            100.0
  Giannis Antetokounmpo  MIL          100.0            100.0
            Rudy Gobert  M

The accolades are working but too many players are hitting the 100 cap — LeBron, Kawhi, Devin Booker and Wembanyama shouldn't be in the same tier. The problem is we're capping `ACCOLADES_RAW` at 100 before scaling, so everyone with a lot of awards hits the ceiling and gets flattened to the same score.
Two fixes needed:
1. Remove the cap of 100 on raw scores — let Jokić's 3 MVPs naturally score higher than Booker's 1 All-NBA selection. The MinMaxScaler will handle the range.
2. Raise the raw score ceiling — let the true spread show through.

In [75]:
def calculate_accolades_score(df):
    """
    Feature 9 — Accolades Score (Weight: 5%)
    
    Same decay formula as before but removing the artificial 100-point cap
    on ACCOLADES_RAW. The cap was flattening everyone with significant
    accolades to the same score — hiding meaningful differences between
    Jokić (3 MVPs + Finals MVP) and Booker (1 All-NBA + 2 All-Stars).
    
    MinMaxScaler handles the normalization to 0-100 after the fact,
    so we don't need a manual cap. Let the raw scores spread naturally
    and the scaler will preserve the relative differences.
    """
    df = df.copy()
    
    award_weights = {
        "mvp"              : 100,
        "finals_mvp"       : 85,
        "all_nba_1"        : 75,
        "all_nba_2"        : 60,
        "all_nba_3"        : 45,
        "all_star"         : 30,
        "dpoy"             : 55,
        "all_def_1"        : 35,
        "all_def_2"        : 20,
        "roy"              : 20,
        "smoy"             : 15,
        "mip"              : 15,
    }
    
    player_accolades = {
        "Nikola Jokic": [
            ("mvp", 2021), ("mvp", 2022), ("mvp", 2024),
            ("finals_mvp", 2023),
            ("all_nba_1", 2022), ("all_nba_1", 2023), ("all_nba_1", 2024),
            ("all_nba_2", 2021),
            ("all_star", 2019), ("all_star", 2020), ("all_star", 2021),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Giannis Antetokounmpo": [
            ("mvp", 2019), ("mvp", 2020),
            ("finals_mvp", 2021),
            ("dpoy", 2020),
            ("all_nba_1", 2019), ("all_nba_1", 2020), ("all_nba_1", 2021),
            ("all_nba_1", 2022), ("all_nba_1", 2023),
            ("all_star", 2018), ("all_star", 2019), ("all_star", 2020),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
            ("all_star", 2024),
        ],
        "LeBron James": [
            ("mvp", 2009), ("mvp", 2010), ("mvp", 2012), ("mvp", 2013),
            ("finals_mvp", 2012), ("finals_mvp", 2013),
            ("finals_mvp", 2016), ("finals_mvp", 2020),
            ("all_nba_1", 2020), ("all_nba_1", 2022),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
        ],
        "Stephen Curry": [
            ("mvp", 2015), ("mvp", 2016),
            ("finals_mvp", 2022),
            ("all_nba_1", 2021), ("all_nba_2", 2022),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
        ],
        "Kevin Durant": [
            ("mvp", 2014),
            ("finals_mvp", 2017), ("finals_mvp", 2018),
            ("all_nba_1", 2021), ("all_nba_2", 2022),
            ("all_star", 2021), ("all_star", 2022), ("all_star", 2023),
        ],
        "Luka Doncic": [
            ("all_nba_1", 2022), ("all_nba_1", 2023), ("all_nba_1", 2024),
            ("all_star", 2021), ("all_star", 2022),
            ("all_star", 2023), ("all_star", 2024),
            ("roy", 2019),
        ],
        "Shai Gilgeous-Alexander": [
            ("mvp", 2025),
            ("all_nba_1", 2024), ("all_nba_1", 2025),
            ("all_star", 2023), ("all_star", 2024), ("all_star", 2025),
        ],
        "Joel Embiid": [
            ("mvp", 2023),
            ("all_nba_1", 2022), ("all_nba_1", 2023),
            ("dpoy", 2023),
            ("all_star", 2021), ("all_star", 2022),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Jayson Tatum": [
            ("finals_mvp", 2024),
            ("all_nba_1", 2023), ("all_nba_1", 2024),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Kawhi Leonard": [
            ("finals_mvp", 2014), ("finals_mvp", 2019),
            ("dpoy", 2015), ("dpoy", 2016),
            ("all_nba_1", 2020),
            ("all_star", 2020), ("all_star", 2021),
        ],
        "James Harden": [
            ("mvp", 2018),
            ("all_nba_1", 2018), ("all_nba_2", 2019),
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Damian Lillard": [
            ("all_nba_1", 2023),
            ("all_star", 2021), ("all_star", 2022),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Anthony Davis": [
            ("dpoy", 2018),
            ("all_nba_1", 2018), ("all_nba_2", 2022),
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Devin Booker": [
            ("all_nba_1", 2022),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Jaylen Brown": [
            ("finals_mvp", 2024),
            ("all_nba_2", 2024),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Donovan Mitchell": [
            ("all_nba_2", 2024),
            ("all_star", 2022), ("all_star", 2023), ("all_star", 2024),
        ],
        "Ja Morant": [
            ("roy", 2020), ("mip", 2022),
            ("all_nba_2", 2022),
            ("all_star", 2022), ("all_star", 2023),
        ],
        "Trae Young": [
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Karl-Anthony Towns": [
            ("roy", 2016),
            ("all_star", 2018), ("all_star", 2019), ("all_star", 2024),
        ],
        "Domantas Sabonis": [
            ("smoy", 2019), ("smoy", 2020),
            ("all_nba_3", 2022),
            ("all_star", 2023), ("all_star", 2024),
        ],
        "Cade Cunningham": [
            ("all_star", 2025),
            ("all_nba_3", 2025),
        ],
        "Tyrese Maxey": [
            ("mip", 2024),
            ("all_star", 2024), ("all_star", 2025),
        ],
        "Jalen Brunson": [
            ("all_star", 2024), ("all_star", 2025),
            ("all_nba_3", 2024),
        ],
        "Anthony Edwards": [
            ("all_star", 2024), ("all_star", 2025),
            ("all_nba_2", 2024),
        ],
        "Victor Wembanyama": [
            ("roy", 2024),
            ("dpoy", 2024),
            ("all_nba_1", 2025),
            ("all_star", 2025),
        ],
        "Tyrese Haliburton": [
            ("all_star", 2024), ("all_star", 2025),
            ("all_nba_3", 2024),
        ],
        "Alperen Sengun": [
            ("all_star", 2025),
        ],
        "Jaren Jackson": [
            ("dpoy", 2023),
            ("all_def_1", 2023), ("all_def_1", 2024),
        ],
        "Bam Adebayo": [
            ("all_star", 2022), ("all_star", 2023),
            ("all_def_1", 2022), ("all_def_1", 2023),
        ],
        "Draymond Green": [
            ("dpoy", 2017),
            ("all_def_1", 2022), ("all_def_1", 2023),
            ("all_star", 2022),
        ],
        "Rudy Gobert": [
            ("dpoy", 2018), ("dpoy", 2019),
            ("dpoy", 2021), ("dpoy", 2022),
            ("all_star", 2021),
        ],
        "Paul George": [
            ("all_nba_2", 2019),
            ("all_star", 2021), ("all_star", 2022),
        ],
        "Zach LaVine": [
            ("all_star", 2022),
        ],
        "De'Aaron Fox": [
            ("all_star", 2023),
        ],
        "LaMelo Ball": [
            ("roy", 2021),
            ("all_star", 2022),
        ],
        "Scottie Barnes": [
            ("roy", 2022),
        ],
        "Paolo Banchero": [
            ("roy", 2023),
            ("all_star", 2025),
        ],
    }
    
    def compute_accolades(player_name, season):
        accolades    = player_accolades.get(player_name, [])
        if not accolades:
            return 0.0
        current_year = int(season.split("-")[0]) + 1
        total        = 0.0
        for award, award_year in accolades:
            weight    = award_weights.get(award, 0)
            years_ago = current_year - award_year
            decay     = 0.5 ** (years_ago / 5)
            total    += weight * decay
        # No cap — let scores spread naturally
        # MinMaxScaler handles normalization to 0-100
        return round(total, 1)
    
    df["ACCOLADES_RAW"] = df.apply(
        lambda row: compute_accolades(row["PLAYER_NAME"], row["SEASON"]),
        axis=1
    )
    
    # Scale to 0-100 — preserves relative differences
    scaler = MinMaxScaler((0, 100))
    df["ACCOLADES_SCORE"] = scaler.fit_transform(
        df[["ACCOLADES_RAW"]]
    ).flatten().round(1)
    
    print(f"✓ Accolades scores calculated")
    print(f"\nRaw score spread:")
    print(f"  Max raw: {df['ACCOLADES_RAW'].max()}")
    print(f"  Min raw (non-zero): {df[df['ACCOLADES_RAW'] > 0]['ACCOLADES_RAW'].min()}")
    
    print(f"\nTop 15 accolades — 2025-26:")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","TEAM","ACCOLADES_RAW","ACCOLADES_SCORE"
    ]].sort_values("ACCOLADES_SCORE", ascending=False)
    .head(15).to_string(index=False))
    
    print(f"\nPlayers with 0 accolades (role players — expected):")
    zero_count = (df[df["SEASON"] == "2025-26"]["ACCOLADES_RAW"] == 0).sum()
    print(f"  {zero_count} players scored 0 — correct for role/unknown tier")
    
    return df

master = calculate_accolades_score(master)

✓ Accolades scores calculated

Raw score spread:
  Max raw: 901.7
  Min raw (non-zero): 11.5

Top 15 accolades — 2025-26:
            PLAYER_NAME TEAM  ACCOLADES_RAW  ACCOLADES_SCORE
           Nikola Jokic  DEN          517.9             57.4
  Giannis Antetokounmpo  MIL          448.1             49.7
Shai Gilgeous-Alexander  OKC          277.8             30.8
            Joel Embiid  PHI          269.6             29.9
           LeBron James  LAL          263.4             29.2
            Luka Doncic  LAL          231.7             25.7
          Stephen Curry  GSW          219.6             24.4
           Kevin Durant  HOU          195.4             21.7
           Jaylen Brown  BOS          152.4             16.9
      Victor Wembanyama  SAS          148.2             16.4
          Kawhi Leonard  LAC          134.7             14.9
            Rudy Gobert  MIN          113.1             12.5
           James Harden  2TM          112.7             12.5
          Anthony Davis 

Now that's a proper spread. Jokić at 57.4, Giannis at 49.7, clear separation all the way down. Wembanyama at 16.4 makes sense — great player but only 2 years of accolades so far. Kawhi at 14.9 reflects the decay on his older awards correctly.
One thing to note — the max raw score of 901.7 is coming from a historical season, not 2025-26. That's LeBron or Giannis in an earlier season when their awards were more recent and worth full decay value. That's the scaler pulling the 2025-26 scores down relative to the historical peak. It's technically correct but means the current season scores look compressed.
We'll revisit this when we build the composite score — the 5% weight keeps accolades from dominating anyway.

### Feature 9 complete — Accolades Score

Removed the artificial 100-point cap on raw scores — was flattening
everyone with significant accolades to the same ceiling.
MinMaxScaler now handles normalization after the full spread is calculated.

Clear separation achieved:
- Jokić 57.4 (3 MVPs + Finals MVP + multiple All-NBA)
- Giannis 49.7 (2 MVPs + Finals MVP + DPOY)
- SGA 30.8 (1 MVP + 2 All-NBA + 3 All-Stars — recent so high decay value)
- Role players: 418 players at 0 — correct

Max raw of 901.7 comes from historical seasons where recent awards
had less decay applied. This is expected behavior — the scaler
normalizes across all seasons so current scores appear compressed
relative to the historical peak. Weights handle this in the composite.

V2: automate by scraping BBRef awards pages for all players.

In [76]:
# Feature 10 — Marketability Score (Placeholder for v2)
# In v1 we use ACCOLADES_SCORE as a proxy for marketability
# since decorated players tend to drive ticket and jersey sales
#
# V2 data sources:
# - Google Trends API (pytrends) — search volume as awareness proxy  
# - Instagram/Twitter followers via SocialBlade
# - National TV appearances from ESPN/TNT broadcast data
#
# For now assign neutral score of 50 for all players
master["MARKETABILITY_SCORE"] = 50.0
print("✓ Marketability score placeholder set (v2 feature)")
print("  All players assigned neutral score of 50.0")

✓ Marketability score placeholder set (v2 feature)
  All players assigned neutral score of 50.0


In [77]:
# Feature 11 — Jersey Sales Score (Placeholder for v2)
# NBA Store publishes top-15 jersey sales annually
# Players outside top 15 get a score derived from Google Trends
#
# V2 data sources:
# - NBA Store annual top-15 list (published each season)
# - Google Trends search volume as proxy for unranked players
#
# For now assign neutral score of 50 for all players
master["JERSEY_SALES_SCORE"] = 50.0
print("✓ Jersey sales score placeholder set (v2 feature)")
print("  All players assigned neutral score of 50.0")

✓ Jersey sales score placeholder set (v2 feature)
  All players assigned neutral score of 50.0


### Features 10 & 11 — Placeholders for v2

Marketability and jersey sales require external data sources
not yet integrated into the pipeline:
- Google Trends API (pytrends library)
- SocialBlade for follower counts
- NBA Store annual jersey rankings

Both assigned neutral score of 50.0 for v1.
Impact is minimal since combined weight is under 10% of final CVI.
The logistic regression will effectively ignore these features
since they have zero variance across all players in v1.

Priority for v2: marketability first (higher weight, more data available)
then jersey sales (lower weight, harder to automate).

In [78]:
# V2 Placeholder — Contract Type Flag
# Player option, team option, fully guaranteed, two-way
# Affects CVI because team options favor the team (good)
# Player options favor the player (slight negative for team)
# Source: Spotrac or BBRef contract pages
master["CONTRACT_TYPE"] = "unknown"
master["CONTRACT_TYPE_SCORE"] = 50.0
print("✓ Contract type placeholder set (v2 feature)")

# V2 Placeholder — Social Media Score  
# Separate from marketability — specifically measures
# engagement rate not just follower count
# Source: Instagram/Twitter API
master["SOCIAL_MEDIA_SCORE"] = 50.0
print("✓ Social media score placeholder set (v2 feature)")

# V2 Placeholder — Injury Risk Score
# Forward-looking availability based on injury TYPE history
# Soft tissue injuries = higher recurrence risk than fractures
# Source: prosportstransactions.com + injury type classification
master["INJURY_RISK_SCORE"] = 50.0
print("✓ Injury risk score placeholder set (v2 feature)")

# V2 Placeholder — Win% Delta
# Did the team improve while this player was on the contract?
# Used as the binary LABEL for logistic regression training
# Source: BBRef team standings by season
master["TEAM_WINPCT_DELTA"] = np.nan
print("✓ Win% delta placeholder set (needed for model labels in nb05)")

✓ Contract type placeholder set (v2 feature)
✓ Social media score placeholder set (v2 feature)
✓ Injury risk score placeholder set (v2 feature)
✓ Win% delta placeholder set (needed for model labels in nb05)


Important note on **TEAM_WINPCT_DELTA** — this one is different from the others. It's not a CVI feature score, it's our binary label for the logistic regression. We need it for notebook 05. It's NaN for now but we need to actually calculate it before training — I'd suggest doing that at the start of notebook 05 rather than here.

### Calculate CVI Score
##### Combine all 9 real feature scores into a single CVI composite score (0-100).
##### This is the final output of feature engineering — one number per player-season that represents overall contract value. Higher = better contract.

In [79]:
def calculate_cvi_composite_score(df):
    """
    Combine all 9 real feature scores into a single CVI composite score (0-100).
    
    This is the final output of feature engineering — one number per player-season
    that represents overall contract value. Higher = better contract.
    
    Weights reflect domain knowledge about what drives contract value:
    - Win impact (19%) — most important, the whole point of a contract is winning
    - Availability (16%) — a star on the bench is worth zero
    - Market comparison (14%) — objective overpay/underpay signal
    - Age/trajectory (12%) — future value on multi-year deals
    - Cap efficiency (11%) — production per dollar of cap space
    - Role fit (10%) — are they earning their minutes
    - Apron impact (9%) — long-term roster flexibility cost
    - Team payroll context (9%) — real apron status from Spotrac
    - Accolades (5%) — pedigree and proven excellence
    
    Marketability and jersey sales excluded from v1 (placeholder 50.0 = neutral,
    would add noise without real data behind them).
    
    Teaching note — why normalize weights to sum to 1.0:
    Our 9 weights sum to 105% (19+16+14+12+11+10+9+9+5 = 105).
    We divide each weight by 1.05 so they sum exactly to 100%.
    This is called normalization — it ensures the composite score
    stays in the 0-100 range without distortion.
    """
    df = df.copy()
    
    # Feature weights — must sum to 1.0 after normalization
    weights = {
        "WIN_IMPACT_SCORE"      : 0.19,
        "AVAILABILITY_SCORE"    : 0.16,
        "MARKET_SCORE"          : 0.14,
        "AGE_SCORE"             : 0.12,
        "CAP_EFFICIENCY_SCORE"  : 0.11,
        "ROLE_FIT_SCORE"        : 0.10,
        "APRON_SCORE"           : 0.09,
        "PAYROLL_CONTEXT_SCORE" : 0.09,
        "ACCOLADES_SCORE"       : 0.05,
    }
    
    # Normalize weights to sum exactly to 1.0
    total_weight = sum(weights.values())
    weights      = {k: v / total_weight for k, v in weights.items()}
    
    print(f"Normalized weights (sum = {sum(weights.values()):.3f}):")
    for feature, weight in weights.items():
        print(f"  {feature:25s}: {weight*100:.1f}%")
    
    # Calculate weighted composite score
    # For each player-season, multiply each feature score by its weight
    # and sum them all together
    df["CVI_SCORE"] = 0.0
    
    for feature, weight in weights.items():
        if feature not in df.columns:
            print(f"  WARNING: {feature} not found — skipping")
            continue
        # Fill NaN feature scores with league median for that feature
        # NaN = missing data, not bad performance — neutral fill is fairest
        median_val = df[feature].median()
        filled     = df[feature].fillna(median_val)
        df["CVI_SCORE"] += filled * weight
    
    df["CVI_SCORE"] = df["CVI_SCORE"].round(1)
    
    # Binary verdict — good / borderline / bad
    # Thresholds based on distribution — top third = good, bottom third = bad
    def get_verdict(score):
        if score >= 62:   return "good"
        elif score >= 45: return "borderline"
        else:             return "bad"
    
    df["CVI_VERDICT"] = df["CVI_SCORE"].apply(get_verdict)
    
    # Flag bonus — playmaker flag adds a small boost
    # Accounts for BPM undervaluing high-AST guards
    df["CVI_SCORE"] = np.where(
        df["PLAYMAKER_FLAG"] == 1,
        (df["CVI_SCORE"] + 2.5).clip(0, 100),
        df["CVI_SCORE"]
    ).round(1)
    
    # Undervalued flag — small boost for star tier franchise-level performers
    df["CVI_SCORE"] = np.where(
        df["UNDERVALUED_FLAG"] == 1,
        (df["CVI_SCORE"] + 2.0).clip(0, 100),
        df["CVI_SCORE"]
    ).round(1)
    
    print(f"\n✓ CVI composite scores calculated")
    print(f"\nScore distribution — 2025-26:")
    scores_2526 = df[df["SEASON"] == "2025-26"]["CVI_SCORE"]
    print(f"  Mean:   {scores_2526.mean():.1f}")
    print(f"  Median: {scores_2526.median():.1f}")
    print(f"  Max:    {scores_2526.max():.1f}")
    print(f"  Min:    {scores_2526.min():.1f}")
    
    print(f"\nVerdict breakdown — 2025-26:")
    print(df[df["SEASON"] == "2025-26"]["CVI_VERDICT"].value_counts().to_string())
    
    print(f"\nTop 15 contracts — 2025-26 (best value):")
    print(df[df["SEASON"] == "2025-26"][[
        "PLAYER_NAME","TEAM","SALARY_M","SALARY_TIER","CVI_SCORE","CVI_VERDICT"
    ]].sort_values("CVI_SCORE", ascending=False)
    .head(15).to_string(index=False))
    
    print(f"\nBottom 15 contracts — 2025-26 (worst value):")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_M"] >= 15)  # only meaningful contracts
    ][["PLAYER_NAME","TEAM","SALARY_M","SALARY_TIER","CVI_SCORE","CVI_VERDICT"]]
    .sort_values("CVI_SCORE")
    .head(15).to_string(index=False))
    
    print(f"\nFranchise tier verdicts — 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_TIER"] == "franchise")
    ][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE","CVI_VERDICT"]]
    .sort_values("CVI_SCORE", ascending=False)
    .to_string(index=False))
    
    return df

master = calculate_cvi_composite_score(master)

Normalized weights (sum = 1.000):
  WIN_IMPACT_SCORE         : 18.1%
  AVAILABILITY_SCORE       : 15.2%
  MARKET_SCORE             : 13.3%
  AGE_SCORE                : 11.4%
  CAP_EFFICIENCY_SCORE     : 10.5%
  ROLE_FIT_SCORE           : 9.5%
  APRON_SCORE              : 8.6%
  PAYROLL_CONTEXT_SCORE    : 8.6%
  ACCOLADES_SCORE          : 4.8%

✓ CVI composite scores calculated

Score distribution — 2025-26:
  Mean:   54.3
  Median: 54.8
  Max:    76.8
  Min:    27.8

Verdict breakdown — 2025-26:
CVI_VERDICT
borderline    284
good           90
bad            76

Top 15 contracts — 2025-26 (best value):
            PLAYER_NAME TEAM  SALARY_M SALARY_TIER  CVI_SCORE CVI_VERDICT
       Payton Pritchard  BOS      7.23        role       76.8        good
      Julian Champagnie  SAS      3.00        role       74.5        good
      Victor Wembanyama  SAS     13.38        role       74.2        good
          Amen Thompson  HOU      9.69        role       73.2        good
          Neemias Que

#### What's working correctly:

SGA at 70.5 as the top franchise player makes perfect sense — elite BPM, under 30, max contract but not the highest salary. Jokić at 65.7 is right behind him. The bottom 15 is genuinely defensible — Middleton at 27.8, Paul George at 30.7, Jimmy Butler at 33.1, Ja Morant, Embiid, Zach LaVine. Every single one of those is a legitimate overpay conversation in basketball circles.

What needs discussion:
Wembanyama scoring 74.2 as a "role" player is the model working correctly but displaying oddly. He's on a rookie contract at $13.38M — the market comparison and cap efficiency features correctly identify him as massively underpaid. But showing him in the role tier looks wrong visually. We should add a "Rookie Scale" flag for this.

Cade Cunningham and Donovan Mitchell in the bad tier as franchise players is worth examining. Cade at 43.6 is likely being hurt by the apron score (Detroit payroll) and market comparison. Mitchell at 40.4 is being killed by Cleveland's second apron status.

Stephen Curry at 38.1 is the most interesting verdict — the model says bad contract, and from a pure value-per-dollar standpoint at 37 years old on $59M that's defensible. But Curry's marketability and jersey sales (both placeholder at 50) would likely push him higher in v2.

#### Adding Rookie Scalee Flag

In [92]:
# Fix 1 — Add rookie scale identification
# Players on rookie contracts are structurally underpaid by design
# Wembanyama, Cade, Scottie Barnes etc. should be flagged separately
# Rookie scale = drafted player still on their first contract
# Proxy: age <= 24 AND salary < $20M AND role tier

master["ROOKIE_SCALE_FLAG"] = (
    (master["AGE"] <= 24) &
    (master["SALARY_M"] < 20) &
    (master["SALARY_TIER"] == "role")
).astype(int)

# Quick check
print("Rookie scale players 2025-26:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["ROOKIE_SCALE_FLAG"] == 1)
][["PLAYER_NAME","TEAM","AGE","SALARY_M","CVI_SCORE"]]
.sort_values("CVI_SCORE", ascending=False)
.head(10).to_string(index=False))

Rookie scale players 2025-26:
      PLAYER_NAME TEAM  AGE  SALARY_M  CVI_SCORE
Julian Champagnie  SAS   24      3.00       81.9
      Jalen Duren  DET   22      6.48       80.0
Victor Wembanyama  SAS   22     13.38       79.5
    Amen Thompson  HOU   23      9.69       78.8
   Moussa Diabate  CHO   24      2.27       77.1
     Oso Ighodaro  PHO   23      1.96       75.8
 Jaime Jaquez Jr.  MIA   24      3.86       75.2
   Ausar Thompson  DET   23      8.78       75.2
 Ryan Kalkbrenner  CHO   24      2.30       74.7
    Cason Wallace  OKC   22      5.82       74.4


In [93]:
# Fix 2 — Adjust verdict thresholds per salary tier
# A role player scoring 62+ is different from a max player scoring 62+
# The bar should be higher for max/franchise contracts

def get_tiered_verdict(row):
    """
    Verdict thresholds adjust based on salary tier.
    We hold max and franchise players to a higher standard
    because their contracts have higher stakes and less margin for error.
    
    Franchise/Max: good >= 60, bad < 42
    Star:          good >= 58, bad < 40  
    Role:          good >= 55, bad < 38
    """
    score = row["CVI_SCORE"]
    tier  = row["SALARY_TIER"]
    
    if tier in ["franchise", "max"]:
        if score >= 60:   return "good"
        elif score >= 42: return "borderline"
        else:             return "bad"
    elif tier == "star":
        if score >= 58:   return "good"
        elif score >= 40: return "borderline"
        else:             return "bad"
    else:  # role + unknown
        if score >= 55:   return "good"
        elif score >= 38: return "borderline"
        else:             return "bad"

master["CVI_VERDICT"] = master.apply(get_tiered_verdict, axis=1)

print("Updated verdict breakdown — 2025-26:")
print(master[master["SEASON"] == "2025-26"]["CVI_VERDICT"].value_counts().to_string())

print("\nFranchise tier with tiered verdicts:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "franchise")
][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE","CVI_VERDICT"]]
.sort_values("CVI_SCORE", ascending=False)
.to_string(index=False))

Updated verdict breakdown — 2025-26:
CVI_VERDICT
good          284
borderline    159
bad             7

Franchise tier with tiered verdicts:
            PLAYER_NAME TEAM  SALARY_M  CVI_SCORE CVI_VERDICT
Shai Gilgeous-Alexander  OKC     38.33       79.9        good
           Nikola Jokic  DEN     55.22       71.0        good
           Tyrese Maxey  PHI     37.96       58.1  borderline
            Luka Doncic  LAL     54.13       54.1  borderline
        Cade Cunningham  DET     46.39       48.3  borderline
       Donovan Mitchell  CLE     46.39       42.3  borderline


In [94]:
# Diagnose Luka and Cade's individual feature scores
diagnose_cols = [
    "PLAYER_NAME", "TEAM", "SALARY_M", "CVI_SCORE",
    "WIN_IMPACT_SCORE", "AVAILABILITY_SCORE", "MARKET_SCORE",
    "AGE_SCORE", "CAP_EFFICIENCY_SCORE", "ROLE_FIT_SCORE",
    "APRON_SCORE", "PAYROLL_CONTEXT_SCORE", "ACCOLADES_SCORE"
]

players_to_check = ["Luka Doncic", "Cade Cunningham", 
                    "Stephen Curry", "Shai Gilgeous-Alexander",
                    "Nikola Jokic"]

print("Feature score breakdown — 2025-26:")
print("=" * 120)
for player in players_to_check:
    row = master[
        (master["PLAYER_NAME"] == player) &
        (master["SEASON"] == "2025-26")
    ][diagnose_cols]
    if len(row) > 0:
        print(f"\n{player}:")
        print(row.to_string(index=False))

Feature score breakdown — 2025-26:

Luka Doncic:
PLAYER_NAME TEAM  SALARY_M  CVI_SCORE  WIN_IMPACT_SCORE  AVAILABILITY_SCORE  MARKET_SCORE  AGE_SCORE  CAP_EFFICIENCY_SCORE  ROLE_FIT_SCORE  APRON_SCORE  PAYROLL_CONTEXT_SCORE  ACCOLADES_SCORE
Luka Doncic  LAL     54.13       54.1              40.4                66.1          29.1       98.0                  37.6            79.0         25.0                   80.0             25.7

Cade Cunningham:
    PLAYER_NAME TEAM  SALARY_M  CVI_SCORE  WIN_IMPACT_SCORE  AVAILABILITY_SCORE  MARKET_SCORE  AGE_SCORE  CAP_EFFICIENCY_SCORE  ROLE_FIT_SCORE  APRON_SCORE  PAYROLL_CONTEXT_SCORE  ACCOLADES_SCORE
Cade Cunningham  DET     46.39       48.3              14.5                71.6          24.1       83.5                  44.2            75.0         35.0                   80.0              7.2

Stephen Curry:
  PLAYER_NAME TEAM  SALARY_M  CVI_SCORE  WIN_IMPACT_SCORE  AVAILABILITY_SCORE  MARKET_SCORE  AGE_SCORE  CAP_EFFICIENCY_SCORE  ROLE_FIT_SCORE 

In [83]:
# Diagnose cap efficiency scores
print("Cap efficiency score distribution 2025-26:")
print(master[master["SEASON"] == "2025-26"]["CAP_EFFICIENCY_SCORE"].describe())

print("\nCap efficiency raw values for our players:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["PLAYER_NAME"].isin([
        "Luka Doncic","Cade Cunningham",
        "Stephen Curry","Shai Gilgeous-Alexander","Nikola Jokic"
    ]))
][["PLAYER_NAME","SALARY_M","SALARY_CAP_PCT",
   "BPM_PERCENTILE","CAP_EFFICIENCY_RAW","CAP_EFFICIENCY_SCORE"]]
.to_string(index=False))

print("\nTop 10 cap efficiency scores 2025-26:")
print(master[master["SEASON"] == "2025-26"][[
    "PLAYER_NAME","SALARY_M","CAP_EFFICIENCY_RAW","CAP_EFFICIENCY_SCORE"
]].sort_values("CAP_EFFICIENCY_SCORE", ascending=False)
.head(10).to_string(index=False))

Cap efficiency score distribution 2025-26:
count    411.000000
mean       0.480535
std        0.706320
min        0.000000
25%        0.100000
50%        0.200000
75%        0.600000
max        7.100000
Name: CAP_EFFICIENCY_SCORE, dtype: float64

Cap efficiency raw values for our players:
            PLAYER_NAME  SALARY_M  SALARY_CAP_PCT  BPM_PERCENTILE  CAP_EFFICIENCY_RAW  CAP_EFFICIENCY_SCORE
        Cade Cunningham     46.39       32.900709            98.7            2.999935                   0.1
            Luka Doncic     54.13       38.390071            99.1            2.581397                   0.1
           Nikola Jokic     55.22       39.163121           100.0            2.553423                   0.1
Shai Gilgeous-Alexander     38.33       27.184397            99.8            3.671224                   0.1
          Stephen Curry     59.61       42.276596            98.1            2.320433                   0.1

Top 10 cap efficiency scores 2025-26:
            PLAYER_NAME

Confirmed — the problem is crystal clear. Myron Gardner has a raw score of 257 while Jokić has 2.55. The MinMaxScaler is compressing everyone into near-zero because minimum salary players making 400K with decent BPM have astronomically high efficiency ratios compared to max players.
The fix is to normalize cap efficiency within salary tier so we're comparing max players to other max players, not to 400K role players:

In [84]:
def recalculate_cap_efficiency_score(df):
    """
    Fixed version of cap efficiency score.
    
    The original version normalized across all salary tiers which
    caused minimum salary players (e.g. Myron Gardner at $400K)
    to have CAP_EFFICIENCY_RAW of 257 while max players scored 2-3.
    MinMaxScaler then compressed all max players to near 0.
    
    Fix: normalize WITHIN salary tier so we compare:
    - Max players vs other max players
    - Star players vs other star players  
    - Role players vs other role players
    
    This is the same peer-normalization approach we used for WIN_IMPACT_SCORE
    and for the same reason — a efficiency ratio only makes sense
    relative to your salary peers, not the entire league.
    
    Teaching note — why this matters:
    A $400K player with 70th percentile BPM has a ratio of ~200.
    A $55M player with 99th percentile BPM has a ratio of ~2.5.
    These aren't comparable on the same scale. Within their tiers
    the $55M player IS being efficient — he's producing at the 99th
    percentile for what max players cost. That's the insight we want.
    """
    df = df.copy()
    
    # NBA salary cap by season
    cap_by_season = {
        "2021-22": 112.4,
        "2022-23": 123.7,
        "2023-24": 136.0,
        "2024-25": 140.6,
        "2025-26": 141.0,
    }
    
    # Salary as % of cap
    df["SALARY_CAP_PCT"] = df.apply(
        lambda row: (row["SALARY_M"] / 
                     cap_by_season.get(row["SEASON"], 141.0)) * 100
        if pd.notna(row["SALARY_M"]) else np.nan,
        axis=1
    )
    
    # BPM percentile within season
    df["BPM_PERCENTILE"] = df.groupby("SEASON")["BPM"].transform(
        lambda x: x.rank(pct=True) * 100
    ).round(1)
    
    # Raw efficiency ratio
    df["CAP_EFFICIENCY_RAW"] = np.where(
        df["SALARY_CAP_PCT"] > 0,
        df["BPM_PERCENTILE"] / df["SALARY_CAP_PCT"],
        np.nan
    )
    
    # Expiring contract bonus
    df["CAP_EFFICIENCY_RAW"] = np.where(
        (df["YEARS_REMAINING"] == 1) & (df["SALARY_M"].notna()),
        df["CAP_EFFICIENCY_RAW"] * 1.10,
        df["CAP_EFFICIENCY_RAW"]
    )
    
    # Normalize WITHIN salary tier — peer comparison
    df["CAP_EFFICIENCY_SCORE"] = np.nan
    
    for tier in ["franchise", "max", "star", "role", "unknown"]:
        mask   = df["SALARY_TIER"] == tier
        values = df.loc[mask, "CAP_EFFICIENCY_RAW"].dropna()
        
        if len(values) < 2:
            continue
        
        scaler = MinMaxScaler((0, 100))
        
        # Only scale rows that have valid values
        valid_mask = mask & df["CAP_EFFICIENCY_RAW"].notna()
        df.loc[valid_mask, "CAP_EFFICIENCY_SCORE"] = scaler.fit_transform(
            df.loc[valid_mask, ["CAP_EFFICIENCY_RAW"]]
        ).flatten().round(1)
    
    print(f"✓ Cap efficiency recalculated with tier normalization")
    print(f"\nCap efficiency by tier — 2025-26:")
    for tier in ["franchise","max","star","role"]:
        tier_data = df[
            (df["SEASON"] == "2025-26") &
            (df["SALARY_TIER"] == tier)
        ]["CAP_EFFICIENCY_SCORE"]
        if len(tier_data) > 0:
            print(f"  {tier:12s}: "
                  f"avg={tier_data.mean():.1f}  "
                  f"min={tier_data.min():.1f}  "
                  f"max={tier_data.max():.1f}")
    
    print(f"\nFranchise tier cap efficiency — 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_TIER"] == "franchise")
    ][["PLAYER_NAME","SALARY_M","BPM_PERCENTILE",
       "SALARY_CAP_PCT","CAP_EFFICIENCY_SCORE"]]
    .sort_values("CAP_EFFICIENCY_SCORE", ascending=False)
    .to_string(index=False))
    
    return df

master = recalculate_cap_efficiency_score(master)

✓ Cap efficiency recalculated with tier normalization

Cap efficiency by tier — 2025-26:
  franchise   : avg=28.3  min=0.0  max=61.3
  max         : avg=59.1  min=0.0  max=100.0
  star        : avg=53.4  min=5.1  max=93.6
  role        : avg=0.6  min=0.0  max=7.1

Franchise tier cap efficiency — 2025-26:
            PLAYER_NAME  SALARY_M  BPM_PERCENTILE  SALARY_CAP_PCT  CAP_EFFICIENCY_SCORE
Shai Gilgeous-Alexander     38.33            99.8       27.184397                  61.3
           Tyrese Maxey     37.96            98.1       26.921986                  59.8
        Cade Cunningham     46.39            98.7       32.900709                  24.5
       Donovan Mitchell     46.39            97.8       32.900709                  23.0
            Luka Doncic     54.13            99.1       38.390071                   1.5
           Nikola Jokic     55.22           100.0       39.163121                   0.0


In [85]:
# Recalculate CVI composite with fixed cap efficiency
master = calculate_cvi_composite_score(master)
master["CVI_VERDICT"] = master.apply(get_tiered_verdict, axis=1)

print("Updated franchise tier verdicts:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "franchise")
][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE","CVI_VERDICT"]]
.sort_values("CVI_SCORE", ascending=False)
.to_string(index=False))

print("\nUpdated feature breakdown for key players:")
for player in ["Luka Doncic","Cade Cunningham",
               "Shai Gilgeous-Alexander","Nikola Jokic"]:
    row = master[
        (master["PLAYER_NAME"] == player) &
        (master["SEASON"] == "2025-26")
    ][["PLAYER_NAME","CVI_SCORE","WIN_IMPACT_SCORE",
       "MARKET_SCORE","CAP_EFFICIENCY_SCORE",
       "APRON_SCORE","PAYROLL_CONTEXT_SCORE"]]
    if len(row) > 0:
        print(f"\n{player}:")
        print(row.to_string(index=False))

Normalized weights (sum = 1.000):
  WIN_IMPACT_SCORE         : 18.1%
  AVAILABILITY_SCORE       : 15.2%
  MARKET_SCORE             : 13.3%
  AGE_SCORE                : 11.4%
  CAP_EFFICIENCY_SCORE     : 10.5%
  ROLE_FIT_SCORE           : 9.5%
  APRON_SCORE              : 8.6%
  PAYROLL_CONTEXT_SCORE    : 8.6%
  ACCOLADES_SCORE          : 4.8%

✓ CVI composite scores calculated

Score distribution — 2025-26:
  Mean:   55.7
  Median: 56.1
  Max:    78.7
  Min:    27.8

Verdict breakdown — 2025-26:
CVI_VERDICT
borderline    275
good          113
bad            62

Top 15 contracts — 2025-26 (best value):
             PLAYER_NAME TEAM  SALARY_M SALARY_TIER  CVI_SCORE CVI_VERDICT
Nickeil Alexander-Walker  ATL     15.16        star       78.7        good
 Shai Gilgeous-Alexander  OKC     38.33   franchise       76.9        good
        Payton Pritchard  BOS      7.23        role       76.8        good
       Julian Champagnie  SAS      3.00        role       74.5        good
       Victor We

Good progress — cap efficiency is now working within tiers. But I see two new issues:

**Issue 1** — Jokić has 0.0 cap efficiency in franchise tier. He's the highest BPM (100th percentile) but the highest salary too. Within the franchise tier he has the worst ratio of production-to-cap-percentage. The scaler is making him the floor. This is mathematically correct but wrong conceptually — the best player in basketball shouldn't score 0 on any meaningful feature.

**Issue 2** — Role tier is still broken (avg 0.6, max 7.1). The role tier still has the Myron Gardner problem internally — minimum salary players with tiny cap percentages dominate the scale.

The root problem is that cap efficiency as a ratio (BPM percentile / salary cap %) fundamentally breaks at salary extremes. Let me rethink this feature entirely:

In [86]:
def recalculate_cap_efficiency_score_v2(df):
    """
    Cap Efficiency Score v2 — complete redesign.
    
    The ratio approach (BPM_percentile / salary_cap_pct) breaks at extremes:
    - Minimum salary players get astronomically high ratios
    - Highest-paid players get penalized for being expensive even when
      they're the best performers (Jokić at $55M with 100th pct BPM = 0)
    
    New approach: within each salary tier, rank players by how much
    their BPM percentile EXCEEDS what their salary percentile predicts.
    
    In other words: given what you're paid relative to your peers,
    are you producing more or less than expected?
    
    This is a within-tier relative measure:
    - A player paid at the 80th percentile of their tier who produces
      at the 95th percentile = positive excess = high score
    - A player paid at the 95th percentile who produces at the 60th
      percentile = negative excess = low score
    
    Teaching note — why this works better:
    We're asking "relative to what you cost, how much do you produce?"
    within your peer group. Jokić costs the most in franchise tier
    BUT produces the most — so his excess production is still positive.
    The old ratio penalized him for being expensive even when justified.
    """
    df = df.copy()
    
    # BPM percentile within season (already calculated but recalc to be safe)
    df["BPM_PERCENTILE"] = df.groupby("SEASON")["BPM"].transform(
        lambda x: x.rank(pct=True) * 100
    ).round(1)
    
    # Salary percentile within tier + season
    df["SALARY_TIER_PERCENTILE"] = df.groupby(
        ["SEASON","SALARY_TIER"]
    )["SALARY_M"].transform(
        lambda x: x.rank(pct=True) * 100
    ).round(1)
    
    # BPM percentile within tier + season
    df["BPM_TIER_PERCENTILE"] = df.groupby(
        ["SEASON","SALARY_TIER"]
    )["BPM"].transform(
        lambda x: x.rank(pct=True) * 100
    ).round(1)
    
    # Excess production = BPM tier percentile - Salary tier percentile
    # Positive = producing more than salary suggests (good)
    # Negative = producing less than salary suggests (bad)
    df["CAP_EFFICIENCY_RAW"] = (
        df["BPM_TIER_PERCENTILE"] - df["SALARY_TIER_PERCENTILE"]
    )
    
    # Expiring contract bonus — creates future cap space
    df["CAP_EFFICIENCY_RAW"] = np.where(
        (df["YEARS_REMAINING"] == 1) & (df["SALARY_M"].notna()),
        df["CAP_EFFICIENCY_RAW"] + 5,  # flat bonus for expiring deals
        df["CAP_EFFICIENCY_RAW"]
    )
    
    # Scale to 0-100 within each tier
    # This preserves relative differences within peers
    df["CAP_EFFICIENCY_SCORE"] = np.nan
    
    for tier in ["franchise","max","star","role","unknown"]:
        mask       = df["SALARY_TIER"] == tier
        valid_mask = mask & df["CAP_EFFICIENCY_RAW"].notna()
        
        if valid_mask.sum() < 2:
            continue
        
        scaler = MinMaxScaler((0, 100))
        df.loc[valid_mask, "CAP_EFFICIENCY_SCORE"] = scaler.fit_transform(
            df.loc[valid_mask, ["CAP_EFFICIENCY_RAW"]]
        ).flatten().round(1)
    
    print(f"✓ Cap efficiency v2 calculated")
    print(f"\nFranchise tier breakdown — 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_TIER"] == "franchise")
    ][["PLAYER_NAME","SALARY_M","BPM_TIER_PERCENTILE",
       "SALARY_TIER_PERCENTILE","CAP_EFFICIENCY_RAW",
       "CAP_EFFICIENCY_SCORE"]]
    .sort_values("CAP_EFFICIENCY_SCORE", ascending=False)
    .to_string(index=False))
    
    print(f"\nCap efficiency by tier averages — 2025-26:")
    for tier in ["franchise","max","star","role"]:
        tier_data = df[
            (df["SEASON"] == "2025-26") &
            (df["SALARY_TIER"] == tier)
        ]["CAP_EFFICIENCY_SCORE"]
        if len(tier_data) > 0:
            print(f"  {tier:12s}: "
                  f"avg={tier_data.mean():.1f}  "
                  f"min={tier_data.min():.1f}  "
                  f"max={tier_data.max():.1f}")
    
    return df

master = recalculate_cap_efficiency_score_v2(master)

✓ Cap efficiency v2 calculated

Franchise tier breakdown — 2025-26:
            PLAYER_NAME  SALARY_M  BPM_TIER_PERCENTILE  SALARY_TIER_PERCENTILE  CAP_EFFICIENCY_RAW  CAP_EFFICIENCY_SCORE
Shai Gilgeous-Alexander     38.33                 83.3                    33.3                50.0                  90.1
           Tyrese Maxey     37.96                 33.3                    16.7                16.6                  63.8
           Nikola Jokic     55.22                100.0                   100.0                 0.0                  50.7
        Cade Cunningham     46.39                 50.0                    58.3                -8.3                  44.2
            Luka Doncic     54.13                 66.7                    83.3               -16.6                  37.6
       Donovan Mitchell     46.39                 16.7                    58.3               -41.6                  17.9

Cap efficiency by tier averages — 2025-26:
  franchise   : avg=50.7  min=17.9  max=9

In [87]:
# Recalculate composite with v2 cap efficiency
master = calculate_cvi_composite_score(master)
master["CVI_VERDICT"] = master.apply(get_tiered_verdict, axis=1)

print("Franchise tier after cap efficiency v2:")
print(master[
    (master["SEASON"] == "2025-26") &
    (master["SALARY_TIER"] == "franchise")
][["PLAYER_NAME","TEAM","SALARY_M","CVI_SCORE","CVI_VERDICT"]]
.sort_values("CVI_SCORE", ascending=False)
.to_string(index=False))

print("\nKey player feature breakdown:")
for player in ["Luka Doncic","Cade Cunningham",
               "Shai Gilgeous-Alexander","Nikola Jokic","Stephen Curry"]:
    row = master[
        (master["PLAYER_NAME"] == player) &
        (master["SEASON"] == "2025-26")
    ][["PLAYER_NAME","CVI_SCORE","WIN_IMPACT_SCORE",
       "MARKET_SCORE","CAP_EFFICIENCY_SCORE",
       "APRON_SCORE","AGE_SCORE","AVAILABILITY_SCORE"]]
    if len(row) > 0:
        print(f"\n{player}:")
        print(row.to_string(index=False))

Normalized weights (sum = 1.000):
  WIN_IMPACT_SCORE         : 18.1%
  AVAILABILITY_SCORE       : 15.2%
  MARKET_SCORE             : 13.3%
  AGE_SCORE                : 11.4%
  CAP_EFFICIENCY_SCORE     : 10.5%
  ROLE_FIT_SCORE           : 9.5%
  APRON_SCORE              : 8.6%
  PAYROLL_CONTEXT_SCORE    : 8.6%
  ACCOLADES_SCORE          : 4.8%

✓ CVI composite scores calculated

Score distribution — 2025-26:
  Mean:   59.5
  Median: 59.8
  Max:    82.9
  Min:    31.9

Verdict breakdown — 2025-26:
CVI_VERDICT
borderline    233
good          184
bad            33

Top 15 contracts — 2025-26 (best value):
             PLAYER_NAME TEAM  SALARY_M SALARY_TIER  CVI_SCORE CVI_VERDICT
        Payton Pritchard  BOS      7.23        role       82.9        good
       Julian Champagnie  SAS      3.00        role       81.9        good
           Neemias Queta  BOS      2.35        role       81.7        good
        Collin Gillespie  PHO      2.30        role       80.6        good
   Sandro Mamuke

This is a massive improvement. Cap efficiency is now working correctly across all tiers — averages around 50 for each tier which is exactly right for a peer-normalized feature. Let me read the results:

#### What's now correct:
SGA at 79.9 is the clear best franchise contract — highest BPM at the lowest salary in his tier. Jokić at 71.0 is right behind — he's the most expensive AND most productive, so his excess production is neutral (0.0 raw) but he scores 50.7 because other franchise players underperform their salary more. 

Luka at 54.1 is now borderline rather than hovering near bad — his market score (29.1) and apron score (25.0) are still dragging him but cap efficiency correctly shows him at 37.6.

Donovan Mitchell at 42.3 bad — this is the model working correctly. He's in franchise tier (great BPM) but on a second apron team (CLE), making $46M, with a low cap efficiency raw of -41.6 (underpaying relative to peers). The CLE apron situation is the killer.

Curry at 43.1 — `AGE_SCORE` of 0.0 and `MARKET_SCORE` of 0.0 are doing most of the damage. The model is saying a 37-year-old making $59.61M is objectively a bad contract on pure production metrics. Hard to argue.

The top 15 is still too role-player heavy. Payton Pritchard, Julian Champagnie, Neemias Queta at the very top is technically correct — they're massively underpaid relative to production — but it makes the output look odd in the app. This is the rookie/minimum salary distortion.

The model is fundamentally sound now. Let's save and move on rather than keep tweaking — we can calibrate thresholds further in notebook 05 when we see the full logistic regression output.

In [88]:
# Save master with all features
os.makedirs("../data/processed", exist_ok=True)
master.to_csv("../data/processed/master_with_features.csv", index=False)

print(f"✓ Saved master_with_features.csv — {len(master)} rows")
print(f"\nFinal column count: {master.shape[1]}")
print(f"\nFeature columns added this notebook:")
feature_cols = [
    "SALARY_TIER","WIN_IMPACT_SCORE","AVAILABILITY_SCORE",
    "MARKET_SCORE","AGE_SCORE","CAP_EFFICIENCY_SCORE",
    "ROLE_FIT_SCORE","APRON_SCORE","PAYROLL_CONTEXT_SCORE",
    "ACCOLADES_SCORE","MARKETABILITY_SCORE","JERSEY_SALES_SCORE",
    "CONTRACT_TYPE_SCORE","INJURY_RISK_SCORE",
    "UNDERVALUED_FLAG","OVERPAID_MAX_FLAG","PLAYMAKER_FLAG",
    "ROOKIE_SCALE_FLAG","CVI_SCORE","CVI_VERDICT"
]
for col in feature_cols:
    exists = "✓" if col in master.columns else "✗ MISSING"
    print(f"  {exists} {col}")

✓ Saved master_with_features.csv — 2230 rows

Final column count: 105

Feature columns added this notebook:
  ✓ SALARY_TIER
  ✓ WIN_IMPACT_SCORE
  ✓ AVAILABILITY_SCORE
  ✓ MARKET_SCORE
  ✓ AGE_SCORE
  ✓ CAP_EFFICIENCY_SCORE
  ✓ ROLE_FIT_SCORE
  ✓ APRON_SCORE
  ✓ PAYROLL_CONTEXT_SCORE
  ✓ ACCOLADES_SCORE
  ✓ MARKETABILITY_SCORE
  ✓ JERSEY_SALES_SCORE
  ✓ CONTRACT_TYPE_SCORE
  ✓ INJURY_RISK_SCORE
  ✓ UNDERVALUED_FLAG
  ✓ OVERPAID_MAX_FLAG
  ✓ PLAYMAKER_FLAG
  ✓ ROOKIE_SCALE_FLAG
  ✓ CVI_SCORE
  ✓ CVI_VERDICT


### Quick Thoughts

On players like donovan mitchell and cade with the teams cap hurting their scores its not really their fault that the team has cap issues. There are actually two completely different questions we can ask:

**Question 1**: Is this a good contract for the TEAM?
Here the apron and payroll context scores make total sense. The team is cap-strapped partly because of this contract. That's a legitimate team-level cost even if the player isn't personally responsible.

**Question 2**: Is this player worth what they're being paid?
Here the apron and payroll context scores are arguably unfair. Mitchell is producing at franchise level. The fact that Cleveland built a problematic roster around him isn't his fault.

Right now the model is answering Question 1 — team contract value. That's actually the right framing for a front office tool. A GM evaluating whether to trade for Mitchell needs to know the full team cost, not just his individual production.

#### Adding new score for player + team value

**CVI_SCORE**          — team perspective (what we have now)

**CVI_PLAYER_SCORE**   — player perspective (individual value only)

The player score would exclude `PAYROLL_CONTEXT_SCORE` and reduce the weight of `APRON_SCORE`, focusing purely on:

- Win impact
- Availability
- Market comparison
- Age/trajectory
- Cap efficiency
- Role fit
- Accolades


#### Donovan Mitchell Example

**Team CVI**:    42.3  🔴  Bad    ← front office view

**Player CVI**:  61.8  🟢  Good   ← player value view

**Gap**:         -19.5  ← team context is hurting his value

- That gap number is actually a really powerful insight on its own — it tells you how much the team situation is suppressing a player's contract value. A large negative gap means the player is good but trapped in a bad roster situation. A positive gap means the team benefits from the player's situation (expiring deal, below apron).
- This would make Mitchell, Cade, and similar players much more nuanced — the model correctly identifies them as good players on teams with cap problems rather than just calling them bad contracts.

### Calculate CVI Player Score
#### - Answers: "Is this player worth what they're being paid?" vs `CVI_SCORE` which answers: "Is this contract good for the team?"

In [90]:
def calculate_cvi_player_score(df):
    """
    CVI Player Score — individual value perspective.
    
    Excludes team-context features (PAYROLL_CONTEXT_SCORE) and
    reduces APRON_SCORE weight since that's partly a team decision.
    
    Answers: "Is this player worth what they're being paid?"
    vs CVI_SCORE which answers: "Is this contract good for the team?"
    
    The gap between the two scores is meaningful:
    Large negative gap = good player, bad team situation (Mitchell, Cade)
    Small gap = player value and team situation are aligned
    Large positive gap = team benefits more than player deserves
    (e.g. expiring deal on a player in decline)
    
    Weights redistribute PAYROLL_CONTEXT_SCORE (8.6%) across the
    remaining individual features proportionally.
    """
    df = df.copy()
    
    # Player-focused weights — no PAYROLL_CONTEXT_SCORE
    # APRON_SCORE kept but at reduced weight (player signs the contract
    # knowing the structure, so some responsibility is theirs)
    player_weights = {
        "WIN_IMPACT_SCORE"   : 0.22,  # increased — most individual metric
        "AVAILABILITY_SCORE" : 0.18,  # increased — player controls this
        "MARKET_SCORE"       : 0.16,  # increased — pure individual value
        "AGE_SCORE"          : 0.13,  # same — individual trajectory
        "CAP_EFFICIENCY_SCORE": 0.12, # same — individual production/cost
        "ROLE_FIT_SCORE"     : 0.10,  # same — individual fit
        "APRON_SCORE"        : 0.05,  # reduced — contract structure
        "ACCOLADES_SCORE"    : 0.04,  # same — individual pedigree
    }
    
    # Normalize to sum to 1.0
    total          = sum(player_weights.values())
    player_weights = {k: v / total for k, v in player_weights.items()}
    
    print("Player score weights (normalized):")
    for feature, weight in player_weights.items():
        print(f"  {feature:25s}: {weight*100:.1f}%")
    
    # Calculate player CVI score
    df["CVI_PLAYER_SCORE"] = 0.0
    
    for feature, weight in player_weights.items():
        if feature not in df.columns:
            print(f"  WARNING: {feature} not found — skipping")
            continue
        median_val = df[feature].median()
        filled     = df[feature].fillna(median_val)
        df["CVI_PLAYER_SCORE"] += filled * weight
    
    df["CVI_PLAYER_SCORE"] = df["CVI_PLAYER_SCORE"].round(1)
    
    # Player verdict uses same tiered thresholds
    df["CVI_PLAYER_VERDICT"] = df.apply(get_tiered_verdict, axis=1)
    
    # Gap score — how much team context helps or hurts
    # Negative = team situation suppressing player value
    # Positive = team situation boosting contract appearance
    df["CVI_GAP"] = (df["CVI_SCORE"] - df["CVI_PLAYER_SCORE"]).round(1)
    
    print(f"\n✓ CVI player scores calculated")
    print(f"\nFranchise tier — team vs player scores 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_TIER"] == "franchise")
    ][["PLAYER_NAME","TEAM","SALARY_M",
       "CVI_SCORE","CVI_PLAYER_SCORE","CVI_GAP",
       "CVI_VERDICT","CVI_PLAYER_VERDICT"]]
    .sort_values("CVI_PLAYER_SCORE", ascending=False)
    .to_string(index=False))
    
    print(f"\nBiggest gaps — team situation hurting player value 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_M"] >= 15)
    ][["PLAYER_NAME","TEAM","SALARY_M",
       "CVI_SCORE","CVI_PLAYER_SCORE","CVI_GAP"]]
    .sort_values("CVI_GAP")
    .head(10).to_string(index=False))
    
    print(f"\nBiggest gaps — team situation helping contract look better 2025-26:")
    print(df[
        (df["SEASON"] == "2025-26") &
        (df["SALARY_M"] >= 15)
    ][["PLAYER_NAME","TEAM","SALARY_M",
       "CVI_SCORE","CVI_PLAYER_SCORE","CVI_GAP"]]
    .sort_values("CVI_GAP", ascending=False)
    .head(10).to_string(index=False))
    
    return df

master = calculate_cvi_player_score(master)

Player score weights (normalized):
  WIN_IMPACT_SCORE         : 22.0%
  AVAILABILITY_SCORE       : 18.0%
  MARKET_SCORE             : 16.0%
  AGE_SCORE                : 13.0%
  CAP_EFFICIENCY_SCORE     : 12.0%
  ROLE_FIT_SCORE           : 10.0%
  APRON_SCORE              : 5.0%
  ACCOLADES_SCORE          : 4.0%

✓ CVI player scores calculated

Franchise tier — team vs player scores 2025-26:
            PLAYER_NAME TEAM  SALARY_M  CVI_SCORE  CVI_PLAYER_SCORE  CVI_GAP CVI_VERDICT CVI_PLAYER_VERDICT
Shai Gilgeous-Alexander  OKC     38.33       79.9              80.6     -0.7        good               good
           Nikola Jokic  DEN     55.22       71.0              71.5     -0.5        good               good
           Tyrese Maxey  PHI     37.96       58.1              54.2      3.9  borderline         borderline
            Luka Doncic  LAL     54.13       54.1              52.9      1.2  borderline         borderline
        Cade Cunningham  DET     46.39       48.3              45.

In [91]:
# Save final master with both scores
master.to_csv("../data/processed/master_with_features.csv", index=False)

print(f"✓ Saved master_with_features.csv — {len(master)} rows")
print(f"\nBoth CVI scores available:")
print(f"  CVI_SCORE        — team perspective")
print(f"  CVI_PLAYER_SCORE — player perspective")
print(f"  CVI_GAP          — difference (negative = team hurting player)")
print(f"\nFinal shape: {master.shape}")

✓ Saved master_with_features.csv — 2230 rows

Both CVI scores available:
  CVI_SCORE        — team perspective
  CVI_PLAYER_SCORE — player perspective
  CVI_GAP          — difference (negative = team hurting player)

Final shape: (2230, 108)


### Dual CVI score system added

Two perspectives on every contract:

`CVI_SCORE` (team view):
Includes all 9 features including PAYROLL_CONTEXT_SCORE.
Answers: "Is this contract good for the team's roster construction?"

`CVI_PLAYER_SCORE` (player view):
Excludes `PAYROLL_CONTEXT_SCORE`, reduces `APRON_SCORE` weight.
Answers: "Is this player worth what they're being paid individually?"

`CVI_GAP` = `CVI_SCORE` - `CVI_PLAYER_SCORE`:

- **Negative gap** = good player trapped in bad team cap situation (Mitchell, Cade — not their fault)
               
- **Positive gap** = team benefits from contract structure (expiring deals, players on cap-friendly teams)
               
- **Near zero gap** = team and player value are well aligned (SGA, Jokić)

- This distinction is critical for trade evaluation:
A team acquiring Mitchell needs to know his PLAYER score (good)
not just his team score (bad) which reflects Cleveland's situation,
not what he brings to a new team.

#### Final Thoughts

#### What's working perfectly:

- SGA and Jokić have near-zero gaps (-0.7 and -0.5) — their individual value and team situation are almost perfectly aligned. That's the model saying "this contract is good from every angle." That's the right verdict for both.
- CLE players (Mitchell, Mobley, Allen) all showing negative gaps — the second apron status is correctly identified as suppressing their individual value scores. Mobley at -4.3 is the biggest gap on the list which makes sense — he's young, productive, but trapped on the most cap-constrained team in the league.
- NYK players (Bridges, KAT, Hart) also showing negative gaps — first apron status hurting them similarly.

#### The "team helping" list is fascinating:
- LeBron at +6.9 means his individual player score (44.3) is actually lower than his team score (51.2) — the LAL being below the first apron makes his contract look better than his pure production justifies at age 40. That's a genuinely insightful finding.
- Khris Middleton at +7.7 gap — his team situation (below apron) is making his terrible contract look slightly less terrible. Remove the team benefit and his player score drops to 24.2. That's brutal and correct.
One thing to flag for notebook 05:
- The gaps are relatively small (most under 5 points) because `PAYROLL_CONTEXT_SCORE` only has 8.6% weight. In the logistic regression we'll want to use both scores as separate features so the model can learn which perspective better predicts contract outcomes.